# BSTabDiff || MiniBooNE || LDHSS || No Permutation || Tuning

In [ ]:
import os

# Suppress Python warnings before importing third-party packages.
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

import gc
import time
import random
import contextlib
import sys

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# PyTorch is still required by BSTabDiff and for CUDA resource reporting.
try:
    import torch
except Exception:
    torch = None

# BSTabDiff
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Global runtime configuration
# ============================================================
GLOBAL_SEED = 42

DATASET_NAME = "MiniBooNE"
CSV_PATH = "MiniBooNE_labeled.csv"
TARGET_COL = "label"
EXPECTED_NUM_CLASSES = 2

# Fixed to CUDA:0 for BSTabDiff generator training
REQUESTED_TORCH_DEVICE_STR = "cuda:0"

OPTUNA_TRIALS = 2000
OPTUNA_STUDY_NAME = "bstabdiff_MiniBooNE_nogobs_cuda0_optuna2000"
OPTUNA_STORAGE = None

# Fixed choices requested
FIXED_PERMUTE_FEATURES = False
FIXED_USE_GOBS = False

optuna.logging.set_verbosity(optuna.logging.ERROR)


# ============================================================
# Small console helper
# ============================================================
def log(msg):
    print(msg, flush=True)


# ============================================================
# Optuna output suppression + summaries
# ============================================================
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def _trial_duration_seconds(trial):
    if trial.datetime_start is None or trial.datetime_complete is None:
        return np.nan
    return (trial.datetime_complete - trial.datetime_start).total_seconds()


def build_optuna_trial_dataframe(study):
    rows = []
    best_so_far = -np.inf

    for t in study.trials:
        value = t.value if t.value is not None else np.nan
        if t.state.name == "COMPLETE" and np.isfinite(value):
            best_so_far = max(best_so_far, value)

        row = {
            "trial": t.number,
            "state": t.state.name,
            "value": value,
            "best_so_far": best_so_far if np.isfinite(best_so_far) else np.nan,
            "duration_sec": _trial_duration_seconds(t),
            "datetime_start": str(t.datetime_start) if t.datetime_start is not None else None,
            "datetime_complete": str(t.datetime_complete) if t.datetime_complete is not None else None,
        }

        for k, v in t.params.items():
            row[f"param__{k}"] = v

        rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    for col in ["value", "best_so_far", "duration_sec"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def get_top_trials_df(trials_df, top_k=10):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    df = df.sort_values(["value", "trial"], ascending=[False, True]).reset_index(drop=True)
    return df.head(top_k)


def get_best_so_far_progression_df(trials_df):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    best_rows = []
    running_best = -np.inf
    for _, row in df.sort_values("trial").iterrows():
        if row["value"] > running_best:
            running_best = row["value"]
            best_rows.append(row)

    if len(best_rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(best_rows).reset_index(drop=True)


def pretty_param_columns(df):
    cols = [c for c in df.columns if c.startswith("param__")]
    return df.rename(columns={c: c.replace("param__", "") for c in cols})


def print_optuna_summary_tables(study, top_k=10):
    trials_df = build_optuna_trial_dataframe(study)
    top_df = get_top_trials_df(trials_df, top_k=top_k)
    best_prog_df = get_best_so_far_progression_df(trials_df)

    log("\n=== Optuna trial summary ===")
    if len(trials_df) == 0:
        log("No trials found.")
        return trials_df, top_df, best_prog_df

    n_complete = int((trials_df["state"] == "COMPLETE").sum())
    n_pruned = int((trials_df["state"] == "PRUNED").sum())
    n_fail = int((trials_df["state"] == "FAIL").sum())

    log(f"Total trials: {len(trials_df)} | COMPLETE: {n_complete} | PRUNED: {n_pruned} | FAIL: {n_fail}")

    if len(top_df) > 0:
        top_show = pretty_param_columns(top_df.copy())
        log(f"\n=== Top {min(top_k, len(top_show))} Optuna trials by objective ===")
        log(top_show.to_markdown(index=False))

    if len(best_prog_df) > 0:
        prog_show = pretty_param_columns(best_prog_df.copy())
        log("\n=== Best-so-far progression ===")
        log(prog_show.to_markdown(index=False))

    return trials_df, top_df, best_prog_df


# ============================================================
# Device / seed helpers
# ============================================================
def resolve_torch_device(device_str: str = REQUESTED_TORCH_DEVICE_STR):
    if torch is None:
        return None
    if not torch.cuda.is_available():
        return torch.device("cpu")
    if not device_str.startswith("cuda:"):
        return torch.device(device_str)
    idx = int(device_str.split(":")[1])
    ndev = torch.cuda.device_count()
    if idx >= ndev:
        raise RuntimeError(
            f"Requested {device_str}, but only {ndev} CUDA device(s) are visible."
        )
    return torch.device(device_str)


if torch is not None:
    DEFAULT_TORCH_DEVICE = resolve_torch_device(REQUESTED_TORCH_DEVICE_STR)
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
else:
    DEFAULT_TORCH_DEVICE = None


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cleanup_torch():
    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    gc.collect()


# ============================================================
# 1) Logistic Regression classifier
# ============================================================
def make_clf(seed=0):
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    n_jobs=None,
                    random_state=seed,
                ),
            ),
        ]
    )


# ============================================================
# 2) Classification metric helpers
# ============================================================
def _predict_proba_binary(clf, X):
    if hasattr(clf, "predict_proba"):
        proba = np.asarray(clf.predict_proba(X), dtype=float)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(clf, "decision_function"):
        scores = np.asarray(clf.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-scores))

    raise ValueError("Classifier has neither predict_proba nor decision_function.")


def _safe_binary_auc_or_acc(clf, X_val, y_val):
    y_val = np.asarray(y_val).astype(int)

    if len(np.unique(y_val)) < 2:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))

    try:
        proba = _predict_proba_binary(clf, X_val)
        return float(roc_auc_score(y_val, proba))
    except Exception:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))


# ============================================================
# 3) Strict 5x5 CV with fold-wise generator training
# ============================================================
def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    generator_fit_fn=None,
    n_syn=200,
    do_trtr=True,
    do_c2st=True,
    seed=0,
):
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided.")

    X_real = np.asarray(X_real)
    y_real = np.asarray(y_real).astype(int)

    classes, counts = np.unique(y_real, return_counts=True)
    if len(classes) != 2:
        raise ValueError(
            f"Binary classification requires exactly 2 classes, but found "
            f"{len(classes)}: {classes.tolist()}"
        )
    if counts.min() < 5:
        raise ValueError(
            "RepeatedStratifiedKFold with n_splits=5 requires at least "
            f"5 samples per class. Class counts: "
            f"{dict(zip(classes.tolist(), counts.tolist()))}"
        )

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)
        X_syn_fold = np.nan_to_num(
            np.asarray(X_syn_fold, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_syn_fold = np.asarray(y_syn_fold).astype(int)

        if len(np.unique(y_syn_fold)) < 2:
            raise RuntimeError(
                f"Fold {fold_id}: the synthetic dataset contains fewer than two classes."
            )

        # TSTR: train on synthetic, test on real.
        clf_tstr = make_clf(seed=fold_seed)
        clf_tstr.fit(X_syn_fold, y_syn_fold)
        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)
        tstr_auc.append(float(roc_auc_score(yte, proba_t)))
        tstr_acc.append(float(accuracy_score(yte, pred_t)))

        # TRTR: train on real, test on real.
        if do_trtr:
            clf_trtr = make_clf(seed=fold_seed)
            clf_trtr.fit(Xtr, ytr)
            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)
            trtr_auc.append(float(roc_auc_score(yte, proba_r)))
            trtr_acc.append(float(accuracy_score(yte, pred_r)))

        # C2ST: distinguish real=1 from synthetic=0.
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)]
            )
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf(seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)
            c2st_auc.append(float(roc_auc_score(ymix_te, proba_m)))
            c2st_acc.append(float(accuracy_score(ymix_te, pred_m)))

        log(
            f"[Fold {fold_id:02d}] "
            f"TSTR ACC={tstr_acc[-1]:.4f}, "
            f"AUC={tstr_auc[-1]:.4f}"
        )
        cleanup_torch()

    def _mean_std(values):
        values = np.asarray(values, dtype=float)
        return float(np.mean(values)), float(np.std(values))

    tstr_auc_mean, tstr_auc_std = _mean_std(tstr_auc)
    tstr_acc_mean, tstr_acc_std = _mean_std(tstr_acc)

    out = {
        "tstr_auc_mean": tstr_auc_mean,
        "tstr_auc_std": tstr_auc_std,
        "tstr_acc_mean": tstr_acc_mean,
        "tstr_acc_std": tstr_acc_std,
        "n_folds": len(tstr_auc),
    }

    if do_trtr and len(trtr_auc) > 0:
        trtr_auc_mean, trtr_auc_std = _mean_std(trtr_auc)
        trtr_acc_mean, trtr_acc_std = _mean_std(trtr_acc)
        out.update(
            {
                "trtr_auc_mean": trtr_auc_mean,
                "trtr_auc_std": trtr_auc_std,
                "trtr_acc_mean": trtr_acc_mean,
                "trtr_acc_std": trtr_acc_std,
            }
        )

    if do_c2st and len(c2st_auc) > 0:
        c2st_auc_mean, c2st_auc_std = _mean_std(c2st_auc)
        c2st_acc_mean, c2st_acc_std = _mean_std(c2st_acc)
        out.update(
            {
                "c2st_auc_mean": c2st_auc_mean,
                "c2st_auc_std": c2st_auc_std,
                "c2st_acc_mean": c2st_acc_mean,
                "c2st_acc_std": c2st_acc_std,
            }
        )

    return out


# ============================================================
# 4) CSV saving helpers
# ============================================================
def save_synth_csv(
    X_syn,
    y_syn=None,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_nogobs_cuda0.csv",
    feature_names=None,
    target_col=TARGET_COL,
):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]
    assert len(feature_names) == m

    df = pd.DataFrame(X_syn, columns=feature_names)
    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        df.insert(0, target_col, y_syn)

    df.to_csv(out_csv, index=False)
    log(
        f"Saved CSV: {out_csv} | shape={df.shape} | "
        f"nan_frac={np.isnan(X_syn).mean():.6g}"
    )
    return out_csv


def save_synth_mask_csv(
    R_syn,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_mask_nogobs_cuda0.csv",
):
    R_syn = np.asarray(R_syn).astype(int)
    dfR = pd.DataFrame(R_syn, columns=[f"r{j}" for j in range(R_syn.shape[1])])
    dfR.to_csv(out_csv, index=False)
    log(
        f"Saved mask CSV: {out_csv} | shape={dfR.shape} | "
        f"observed_frac={R_syn.mean():.6g}"
    )
    return out_csv


# ============================================================
# 5) BSTabDiff Optuna tuning
# ============================================================
def _sample_bstabdiff_params(trial):
    params = {}

    params["M"] = trial.suggest_categorical("M", [8, 12, 16, 24, 32, 48, 64])
    params["prior_type"] = trial.suggest_categorical("prior_type", ["diffusion", "flow"])
    params["prior_epochs"] = trial.suggest_categorical("prior_epochs", [300, 500, 800, 1200, 1600])
    params["prior_batch"] = trial.suggest_categorical("prior_batch", [16, 32, 64, 128])
    params["prior_lr"] = trial.suggest_float("prior_lr", 5e-5, 5e-3, log=True)
    params["use_ema"] = True
    params["ema_decay"] = trial.suggest_float("ema_decay", 0.99, 0.9999, log=True)

    # Fixed: no GO-BS, no permutation
    params["use_gobs"] = FIXED_USE_GOBS
    params["permute_features"] = FIXED_PERMUTE_FEATURES

    # placeholders for compatibility
    params["gobs_num_clusters"] = 7
    params["gobs_metric"] = "correlation"
    params["gobs_bins"] = 32
    params["gobs_top_k"] = None
    params["gobs_refine_order"] = False
    params["gobs_direction_select"] = False
    params["gobs_refine_passes"] = 0
    params["gobs_boundary_refine_passes"] = 1
    params["gobs_boundary_window"] = 8
    params["gobs_lambda_cross"] = 1.0
    params["gobs_gamma_balance"] = 1e-6

    params["n_syn_inner"] = trial.suggest_categorical("n_syn_inner", [64, 100, 128, 200, 256])

    return params


class _FrozenTrialAdapter:
    def __init__(self, params_dict):
        self.params_dict = params_dict

    def suggest_categorical(self, name, choices):
        if name in self.params_dict:
            return self.params_dict[name]
        return choices[0]

    def suggest_int(self, name, low, high, step=1, log=False):
        if name in self.params_dict:
            return int(self.params_dict[name])
        return low

    def suggest_float(self, name, low, high, step=None, log=False):
        if name in self.params_dict:
            return float(self.params_dict[name])
        return low


def _fit_bstabdiff_with_params(
    Xtr,
    ytr,
    params,
    device=REQUESTED_TORCH_DEVICE_STR,
    seed=0,
    verbose_every=0,
):
    feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(Xtr.shape[1])]

    gen = fit_block_subunit_generator(
        X=Xtr,
        feature_specs=feature_specs,
        y=ytr,
        M=params["M"],
        blocks=None,
        permute_features=False,
        prior_type=params["prior_type"],
        device=device,
        seed=seed,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=verbose_every,
        save_dir=None,
        save_name="bstabdiff_MiniBooNE_nogobs_cuda0",
        save_best=True,
        use_ema=params["use_ema"],
        ema_decay=params["ema_decay"],
        return_train_info=False,
        use_gobs=False,
        gobs_num_clusters=params["gobs_num_clusters"],
        gobs_metric=params["gobs_metric"],
        gobs_bins=params["gobs_bins"],
        gobs_top_k=params["gobs_top_k"],
        gobs_refine_order=params["gobs_refine_order"],
        gobs_direction_select=params["gobs_direction_select"],
        gobs_refine_passes=params["gobs_refine_passes"],
        gobs_boundary_refine_passes=params["gobs_boundary_refine_passes"],
        gobs_boundary_window=params["gobs_boundary_window"],
        gobs_lambda_cross=params["gobs_lambda_cross"],
        gobs_gamma_balance=params["gobs_gamma_balance"],
        gobs_use_cpu_kmeans=False,
    )
    return gen


def tune_bstabdiff_once(
    X,
    y,
    device=REQUESTED_TORCH_DEVICE_STR,
    n_trials=OPTUNA_TRIALS,
    seed=GLOBAL_SEED,
    inner_val_size=0.2,
    show_progress_bar=True,
):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).astype(int)

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=inner_val_size,
        random_state=seed,
    )
    (itr, iva), = splitter.split(X, y)
    X_inner_tr, y_inner_tr = X[itr], y[itr]
    X_inner_va, y_inner_va = X[iva], y[iva]

    def objective(trial):
        params = _sample_bstabdiff_params(trial)
        trial_seed = int(seed + 1000 + trial.number)

        try:
            gen_trial = _fit_bstabdiff_with_params(
                Xtr=X_inner_tr,
                ytr=y_inner_tr,
                params=params,
                device=device,
                seed=trial_seed,
                verbose_every=0,
            )

            X_syn, R_syn, y_syn = gen_trial.sample(n=int(params["n_syn_inner"]))
            X_syn = np.nan_to_num(np.asarray(X_syn, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_syn = np.asarray(y_syn).astype(int)

            if len(np.unique(y_syn)) < 2:
                return 0.0

            proxy_clf = make_clf(seed=trial_seed)
            proxy_clf.fit(X_syn, y_syn)
            score = _safe_binary_auc_or_acc(
                proxy_clf,
                X_inner_va,
                y_inner_va,
            )

            score_adj = float(score) - 1e-4 * float(params["M"])
            return score_adj

        except Exception:
            return 0.0

        finally:
            cleanup_torch()

    sampler = TPESampler(
        seed=seed,
        multivariate=True,
        warn_independent_sampling=False,
    )
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=OPTUNA_STUDY_NAME,
        storage=OPTUNA_STORAGE,
        load_if_exists=False,
    )

    with suppress_stdout_stderr():
        study.optimize(objective, n_trials=n_trials, show_progress_bar=show_progress_bar)

    best_full = _sample_bstabdiff_params(_FrozenTrialAdapter(dict(study.best_trial.params)))
    best_full["permute_features"] = False
    best_full["use_gobs"] = False

    trials_df, top_trials_df, best_progression_df = print_optuna_summary_tables(study, top_k=10)
    return best_full, study, trials_df, top_trials_df, best_progression_df


def make_generator_fit_fn_from_best_params(
    best_params,
    device=REQUESTED_TORCH_DEVICE_STR,
    verbose_every=0,
):
    def _fit(Xtr, ytr, seed=0):
        gen = _fit_bstabdiff_with_params(
            Xtr=np.asarray(Xtr, dtype=np.float32),
            ytr=np.asarray(ytr).astype(int),
            params=best_params,
            device=device,
            seed=seed,
            verbose_every=verbose_every,
        )
        return gen
    return _fit


# ============================================================
# 6) Fidelity metric helpers
# ============================================================
def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return 0.0, 0.0

    values = np.concatenate([x, y])
    labels = np.concatenate([np.zeros(nx, dtype=int), np.ones(ny, dtype=int)])
    order = np.argsort(values)
    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1
    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny
    diff = cdf1 - cdf2

    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _rank_transform(X):
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    ranks = np.empty_like(X, dtype=float)
    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=float)
    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centered = X - mean
    var = (centered**2).mean(axis=0)
    eps = 1e-8
    std = np.sqrt(var + eps)
    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)
    return mean, var, skew, kurt


def compute_fidelity_metrics(
    X_real,
    y_real,
    X_syn,
    y_syn,
    feature_names=None,
    generator=None,
    random_state=0,
):
    X_real = np.asarray(X_real, dtype=float)
    X_syn = np.asarray(X_syn, dtype=float)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."
    n_features = X_real.shape[1]

    ks_stats = np.zeros(n_features, dtype=float)
    w1_stats = np.zeros(n_features, dtype=float)
    for j in range(n_features):
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    pearson_real = np.corrcoef(X_real, rowvar=False)
    pearson_syn = np.corrcoef(X_syn, rowvar=False)
    mask_offdiag = ~np.eye(n_features, dtype=bool)
    pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]
    pearson_mean = float(np.mean(pearson_abs_diff))
    pearson_max = float(np.max(pearson_abs_diff))

    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)
    spearman_real = np.corrcoef(ranks_real, rowvar=False)
    spearman_syn = np.corrcoef(ranks_syn, rowvar=False)
    spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]
    spearman_mean = float(np.mean(spearman_abs_diff))
    spearman_max = float(np.max(spearman_abs_diff))

    mi_real = mi_syn = mi_abs = None
    mi_mean = mi_max = None
    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)
        mi_real = mutual_info_classif(X_real, y_real, random_state=random_state)
        mi_syn = mutual_info_classif(X_syn, y_syn, random_state=random_state)
        mi_abs = np.abs(mi_real - mi_syn)
        mi_mean = float(np.mean(mi_abs))
        mi_max = float(np.max(mi_abs))

    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)
    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    ll_real = ll_syn = None
    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    nn_stats = {}
    try:
        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(X_real)
        dists, idx = nn.kneighbors(X_syn, return_distance=True)
        dists = dists.ravel()
        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_real": mi_real,
                "mi_syn": mi_syn,
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )
    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# 7) Load MiniBooNE dataset
# ============================================================
def load_miniboone_dataset(
    csv_path=CSV_PATH,
    label_col=TARGET_COL,
    expected_num_classes=EXPECTED_NUM_CLASSES,
):
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. "
            f"Columns: {list(df.columns)[:10]} ..."
        )

    y_raw = df[label_col].to_numpy()
    X_df = df.drop(columns=[label_col])
    X_df = X_df.select_dtypes(include=[np.number])

    if X_df.shape[1] == 0:
        raise ValueError("No numeric feature columns were found in the dataset.")

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Encode arbitrary binary labels as 0 and 1.
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, class_names = pd.factorize(y_raw, sort=True)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, class_names = pd.factorize(y_raw, sort=True)
        else:
            y = y_raw.astype(int, copy=False)
            class_names = unique

    y = np.asarray(y, dtype=int)
    classes, counts = np.unique(y, return_counts=True)

    if expected_num_classes is not None and len(classes) != expected_num_classes:
        raise ValueError(
            f"Expected {expected_num_classes} classes, but found {len(classes)}: "
            f"{classes.tolist()}"
        )

    if counts.min() < 5:
        raise ValueError(
            "The requested 5-fold stratified CV requires at least 5 samples "
            f"per class. Class counts: {dict(zip(classes.tolist(), counts.tolist()))}"
        )

    return X, y, list(X_df.columns), np.asarray(class_names)


# ============================================================
# 8) Main
# ============================================================
if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    log(f"Using torch device: {DEFAULT_TORCH_DEVICE}")
    log(f"Fixed permute_features: {FIXED_PERMUTE_FEATURES}")
    log(f"Fixed use_gobs: {FIXED_USE_GOBS}")
    log("Classifier: Logistic Regression only")

    # -------- Load MiniBooNE dataset --------
    X, y, feature_names, original_class_names = load_miniboone_dataset(
        csv_path=CSV_PATH,
        label_col=TARGET_COL,
        expected_num_classes=EXPECTED_NUM_CLASSES,
    )
    class_counts = dict(zip(*np.unique(y, return_counts=True)))
    log(
        f"Loaded {DATASET_NAME}: X.shape={X.shape}, y.shape={y.shape}, "
        f"encoded_classes={np.unique(y).tolist()}, class_counts={class_counts}"
    )
    log(f"Original class values: {original_class_names.tolist()}")

    # ============================================================
    # (0) ONE-TIME Optuna tuning on the full MiniBooNE dataset
    # ============================================================
    log(
        f"\n=== Running ONE-TIME Optuna tuning for BSTabDiff "
        f"({OPTUNA_TRIALS} trials, no GO-BS, no permutation, cuda:0) "
        f"on full {DATASET_NAME} ==="
    )
    t_tune0 = time.time()

    best_bstabdiff_params, study, trials_df, top_trials_df, best_progression_df = tune_bstabdiff_once(
        X=X,
        y=y,
        device=REQUESTED_TORCH_DEVICE_STR,
        n_trials=OPTUNA_TRIALS,
        seed=GLOBAL_SEED,
        inner_val_size=0.2,
        show_progress_bar=True,
    )

    t_tune = time.time() - t_tune0

    log("\n=== Best BSTabDiff params from single Optuna study ===")
    log(str(best_bstabdiff_params))
    log(f"Best Optuna objective value: {study.best_value:.6f}")
    log(f"Tuning time: {t_tune:.2f} sec")

    # Save trial-wise summaries.
    trials_csv = "bstabdiff_MiniBooNE_optuna_trials_full_nogobs_cuda0.csv"
    top_trials_csv = "bstabdiff_MiniBooNE_optuna_top_trials_nogobs_cuda0.csv"
    progression_csv = "bstabdiff_MiniBooNE_optuna_best_progression_nogobs_cuda0.csv"

    trials_df.to_csv(trials_csv, index=False)
    top_trials_df.to_csv(top_trials_csv, index=False)
    best_progression_df.to_csv(progression_csv, index=False)

    log("\nSaved Optuna summaries:")
    log(f"  - {trials_csv}")
    log(f"  - {top_trials_csv}")
    log(f"  - {progression_csv}")

    generator_fit_fn = make_generator_fit_fn_from_best_params(
        best_params=best_bstabdiff_params,
        device=REQUESTED_TORCH_DEVICE_STR,
        verbose_every=0,
    )

    # ============================================================
    # (A) Fit BSTabDiff once on full MiniBooNE data for reporting
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass

    cpu_before = proc.memory_info().rss if proc is not None else 0
    t0 = time.time()
    gen_full = generator_fit_fn(X, y, seed=GLOBAL_SEED)
    gen_train_time = time.time() - t0
    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)

    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated(device=DEFAULT_TORCH_DEVICE)
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024**3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024**3) if cpu_peak > 0 else 0.0

    log(f"\n=== BSTabDiff generator: resource usage (trained once on full {DATASET_NAME}) ===")
    log(f"  Train time: {gen_train_time:.2f} sec")
    log(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    log(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    # -------- Sample one synthetic dataset --------
    N_SYN = 200
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)

    synth_csv = "MiniBooNE_synth_BSTabDiff_n200_nogobs_cuda0.csv"
    mask_csv = "MiniBooNE_synth_BSTabDiff_n200_mask_nogobs_cuda0.csv"

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv=synth_csv,
        feature_names=feature_names,
        target_col=TARGET_COL,
    )
    save_synth_mask_csv(
        R_syn_global,
        out_csv=mask_csv,
    )

    # ============================================================
    # (B) 5x5 CV: strict fold-wise BSTabDiff + Logistic Regression
    # ============================================================
    log(
        "\n=== Evaluating STRICT BSTabDiff synthetic + Logistic Regression "
        "(5x5 TSTR/TRTR/C2ST) ==="
    )

    res = eval_5x5_tstr_foldwise_gen(
        X_real=X,
        y_real=y,
        generator_fit_fn=generator_fit_fn,
        n_syn=N_SYN,
        do_trtr=True,
        do_c2st=True,
        seed=0,
    )

    log(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
    log(
        f"  TSTR AUC: "
        f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}"
    )
    log(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
    log(
        f"  TRTR AUC: "
        f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}"
    )
    log(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
    log(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

    summary_df = pd.DataFrame(
        [
            {
                "Classifier": "Logistic Regression",
                "TSTR ACC (mean ± std)": (
                    f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}"
                ),
                "TSTR AUC (mean ± std)": (
                    f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}"
                ),
                "TRTR ACC (mean ± std)": (
                    f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}"
                ),
                "TRTR AUC (mean ± std)": (
                    f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}"
                ),
                "C2ST ACC (mean ± std)": (
                    f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}"
                ),
                "C2ST AUC (mean ± std)": (
                    f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}"
                ),
            }
        ]
    )

    summary_csv = "bstabdiff_MiniBooNE_cv_summary_lr_nogobs_cuda0.csv"
    summary_df.to_csv(summary_csv, index=False)

    log(
        f"\n=== {DATASET_NAME}: STRICT BSTabDiff synthetic + "
        "Logistic Regression: 5x5 TSTR/TRTR/C2ST summary ==="
    )
    log(summary_df.to_markdown(index=False))
    log(f"Saved CV summary: {summary_csv}")

    # ============================================================
    # (C) Fidelity diagnostics
    # ============================================================
    fidelity = compute_fidelity_metrics(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        feature_names=feature_names,
        generator=gen_full,
        random_state=0,
    )

    log(f"\n=== {DATASET_NAME}: Fidelity metrics (real vs synthetic) ===")
    ks_mean = float(np.mean(fidelity["ks_per_feature"]))
    ks_max = float(np.max(fidelity["ks_per_feature"]))
    w1_mean = float(np.mean(fidelity["wasserstein_per_feature"]))
    w1_max = float(np.max(fidelity["wasserstein_per_feature"]))
    log(f"  KS statistic per feature:    mean={ks_mean:.4f}, max={ks_max:.4f}")
    log(f"  Wasserstein-1 per feature:   mean={w1_mean:.4f}, max={w1_max:.4f}")
    log(
        f"  Pearson |corr_real - corr_syn|: "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )
    log(
        f"  Spearman |corr_real - corr_syn|: "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]
    log("  Higher-order moments (mean abs diff over features):")
    log(f"    mean:  {higher['mean_abs_diff']:.4e}")
    log(f"    var:   {higher['var_abs_diff']:.4e}")
    log(f"    skew:  {higher['skew_abs_diff']:.4e}")
    log(f"    kurt:  {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        log(
            f"  Mutual information (feature-label) |ΔMI|: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    if "loglik_real_mean" in fidelity:
        log(
            f"  Log-likelihood (if available): "
            f"real={fidelity['loglik_real_mean']}, "
            f"synthetic={fidelity['loglik_syn_mean']}"
        )

    log(
        f"  NaN fraction (real): {fidelity['nan_frac_real']:.4e}, "
        f"NaN fraction (synthetic): {fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]
        log("  Privacy (NN distance from synthetic to nearest real):")
        log(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

Using torch device: cuda:0
Fixed permute_features: False
Fixed use_gobs: False
Classifier: Logistic Regression only
Loaded MiniBooNE: X.shape=(130064, 50), y.shape=(130064,), encoded_classes=[0, 1], class_counts={np.int64(0): np.int64(93565), np.int64(1): np.int64(36499)}
Original class values: [0, 1]

=== Running ONE-TIME Optuna tuning for BSTabDiff (2000 trials, no GO-BS, no permutation, cuda:0) on full MiniBooNE ===


  0%|          | 0/2000 [00:00<?, ?it/s]


=== Optuna trial summary ===
Total trials: 2000 | COMPLETE: 2000 | PRUNED: 0 | FAIL: 0

=== Top 10 Optuna trials by objective ===
|   trial | state    |    value |   best_so_far |   duration_sec | datetime_start             | datetime_complete          |   M | prior_type   |   prior_epochs |   prior_batch |    prior_lr |   ema_decay |   n_syn_inner |
|--------:|:---------|---------:|--------------:|---------------:|:---------------------------|:---------------------------|----:|:-------------|---------------:|--------------:|------------:|------------:|--------------:|
|    1645 | COMPLETE | 0.940885 |      0.940885 |        4.64142 | 2026-07-25 08:59:45.495096 | 2026-07-25 08:59:50.136513 |  64 | diffusion    |            800 |            64 | 0.000921878 |    0.990843 |           256 |
|     427 | COMPLETE | 0.940078 |      0.940078 |        6.76493 | 2026-07-25 06:00:05.935489 | 2026-07-25 06:00:12.700418 |  48 | diffusion    |           1600 |           128 | 8.66174e-05 |    0.99

# BSTabDiff || MiniBooNE || LDHSS || GO-BS || Tuning

In [ ]:
import os

# Suppress Python warnings before importing third-party packages.
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

import gc
import time
import random
import contextlib
import sys

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# PyTorch is still required by BSTabDiff and for CUDA resource reporting.
try:
    import torch
except Exception:
    torch = None

# BSTabDiff
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Global runtime configuration
# ============================================================
GLOBAL_SEED = 42

DATASET_NAME = "MiniBooNE"
CSV_PATH = "MiniBooNE_labeled.csv"
TARGET_COL = "label"
EXPECTED_NUM_CLASSES = 2

# Fixed to CUDA:0 for BSTabDiff generator training
REQUESTED_TORCH_DEVICE_STR = "cuda:0"

# Tune GO-BS only; all previously optimized BSTabDiff settings are fixed.
OPTUNA_TRIALS = 150
OPTUNA_STUDY_NAME = "bstabdiff_MiniBooNE_gobs_cuda0_optuna150"
OPTUNA_STORAGE = None

FIXED_PERMUTE_FEATURES = False
FIXED_USE_GOBS = True

# Best MiniBooNE BSTabDiff settings from the completed no-GO-BS study.
# These remain fixed in every one of the 150 GO-BS trials.
FIXED_BSTABDIFF_PARAMS = {
    "M": 64,
    "prior_type": "diffusion",
    "prior_epochs": 800,
    "prior_batch": 64,
    "prior_lr": 0.000921878160004023,
    "use_ema": True,
    "ema_decay": 0.9908430906937514,
    "n_syn_inner": 256,
}

optuna.logging.set_verbosity(optuna.logging.ERROR)


# ============================================================
# Small console helper
# ============================================================
def log(msg):
    print(msg, flush=True)


# ============================================================
# Optuna output suppression + summaries
# ============================================================
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def _trial_duration_seconds(trial):
    if trial.datetime_start is None or trial.datetime_complete is None:
        return np.nan
    return (trial.datetime_complete - trial.datetime_start).total_seconds()


def build_optuna_trial_dataframe(study):
    rows = []
    best_so_far = -np.inf

    for t in study.trials:
        value = t.value if t.value is not None else np.nan
        if t.state.name == "COMPLETE" and np.isfinite(value):
            best_so_far = max(best_so_far, value)

        row = {
            "trial": t.number,
            "state": t.state.name,
            "value": value,
            "best_so_far": best_so_far if np.isfinite(best_so_far) else np.nan,
            "duration_sec": _trial_duration_seconds(t),
            "datetime_start": str(t.datetime_start) if t.datetime_start is not None else None,
            "datetime_complete": str(t.datetime_complete) if t.datetime_complete is not None else None,
        }

        for k, v in t.params.items():
            row[f"param__{k}"] = v

        rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    for col in ["value", "best_so_far", "duration_sec"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def get_top_trials_df(trials_df, top_k=10):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    df = df.sort_values(["value", "trial"], ascending=[False, True]).reset_index(drop=True)
    return df.head(top_k)


def get_best_so_far_progression_df(trials_df):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    best_rows = []
    running_best = -np.inf
    for _, row in df.sort_values("trial").iterrows():
        if row["value"] > running_best:
            running_best = row["value"]
            best_rows.append(row)

    if len(best_rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(best_rows).reset_index(drop=True)


def pretty_param_columns(df):
    cols = [c for c in df.columns if c.startswith("param__")]
    return df.rename(columns={c: c.replace("param__", "") for c in cols})


def print_optuna_summary_tables(study, top_k=10):
    trials_df = build_optuna_trial_dataframe(study)
    top_df = get_top_trials_df(trials_df, top_k=top_k)
    best_prog_df = get_best_so_far_progression_df(trials_df)

    log("\n=== Optuna trial summary ===")
    if len(trials_df) == 0:
        log("No trials found.")
        return trials_df, top_df, best_prog_df

    n_complete = int((trials_df["state"] == "COMPLETE").sum())
    n_pruned = int((trials_df["state"] == "PRUNED").sum())
    n_fail = int((trials_df["state"] == "FAIL").sum())

    log(f"Total trials: {len(trials_df)} | COMPLETE: {n_complete} | PRUNED: {n_pruned} | FAIL: {n_fail}")

    if len(top_df) > 0:
        top_show = pretty_param_columns(top_df.copy())
        log(f"\n=== Top {min(top_k, len(top_show))} Optuna trials by objective ===")
        log(top_show.to_markdown(index=False))

    if len(best_prog_df) > 0:
        prog_show = pretty_param_columns(best_prog_df.copy())
        log("\n=== Best-so-far progression ===")
        log(prog_show.to_markdown(index=False))

    return trials_df, top_df, best_prog_df


# ============================================================
# Device / seed helpers
# ============================================================
def resolve_torch_device(device_str: str = REQUESTED_TORCH_DEVICE_STR):
    if torch is None:
        return None
    if not torch.cuda.is_available():
        return torch.device("cpu")
    if not device_str.startswith("cuda:"):
        return torch.device(device_str)
    idx = int(device_str.split(":")[1])
    ndev = torch.cuda.device_count()
    if idx >= ndev:
        raise RuntimeError(
            f"Requested {device_str}, but only {ndev} CUDA device(s) are visible."
        )
    return torch.device(device_str)


if torch is not None:
    DEFAULT_TORCH_DEVICE = resolve_torch_device(REQUESTED_TORCH_DEVICE_STR)
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
else:
    DEFAULT_TORCH_DEVICE = None


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cleanup_torch():
    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    gc.collect()


# ============================================================
# 1) Logistic Regression classifier
# ============================================================
def make_clf(seed=0):
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    n_jobs=None,
                    random_state=seed,
                ),
            ),
        ]
    )


# ============================================================
# 2) Classification metric helpers
# ============================================================
def _predict_proba_binary(clf, X):
    if hasattr(clf, "predict_proba"):
        proba = np.asarray(clf.predict_proba(X), dtype=float)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(clf, "decision_function"):
        scores = np.asarray(clf.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-scores))

    raise ValueError("Classifier has neither predict_proba nor decision_function.")


def _safe_binary_auc_or_acc(clf, X_val, y_val):
    y_val = np.asarray(y_val).astype(int)

    if len(np.unique(y_val)) < 2:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))

    try:
        proba = _predict_proba_binary(clf, X_val)
        return float(roc_auc_score(y_val, proba))
    except Exception:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))


# ============================================================
# 3) Strict 5x5 CV with fold-wise generator training
# ============================================================
def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    generator_fit_fn=None,
    n_syn=200,
    do_trtr=True,
    do_c2st=True,
    seed=0,
):
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided.")

    X_real = np.asarray(X_real)
    y_real = np.asarray(y_real).astype(int)

    classes, counts = np.unique(y_real, return_counts=True)
    if len(classes) != 2:
        raise ValueError(
            f"Binary classification requires exactly 2 classes, but found "
            f"{len(classes)}: {classes.tolist()}"
        )
    if counts.min() < 5:
        raise ValueError(
            "RepeatedStratifiedKFold with n_splits=5 requires at least "
            f"5 samples per class. Class counts: "
            f"{dict(zip(classes.tolist(), counts.tolist()))}"
        )

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)
        X_syn_fold = np.nan_to_num(
            np.asarray(X_syn_fold, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_syn_fold = np.asarray(y_syn_fold).astype(int)

        if len(np.unique(y_syn_fold)) < 2:
            raise RuntimeError(
                f"Fold {fold_id}: the synthetic dataset contains fewer than two classes."
            )

        # TSTR: train on synthetic, test on real.
        clf_tstr = make_clf(seed=fold_seed)
        clf_tstr.fit(X_syn_fold, y_syn_fold)
        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)
        tstr_auc.append(float(roc_auc_score(yte, proba_t)))
        tstr_acc.append(float(accuracy_score(yte, pred_t)))

        # TRTR: train on real, test on real.
        if do_trtr:
            clf_trtr = make_clf(seed=fold_seed)
            clf_trtr.fit(Xtr, ytr)
            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)
            trtr_auc.append(float(roc_auc_score(yte, proba_r)))
            trtr_acc.append(float(accuracy_score(yte, pred_r)))

        # C2ST: distinguish real=1 from synthetic=0.
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)]
            )
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf(seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)
            c2st_auc.append(float(roc_auc_score(ymix_te, proba_m)))
            c2st_acc.append(float(accuracy_score(ymix_te, pred_m)))

        log(
            f"[Fold {fold_id:02d}] "
            f"TSTR ACC={tstr_acc[-1]:.4f}, "
            f"AUC={tstr_auc[-1]:.4f}"
        )
        cleanup_torch()

    def _mean_std(values):
        values = np.asarray(values, dtype=float)
        return float(np.mean(values)), float(np.std(values))

    tstr_auc_mean, tstr_auc_std = _mean_std(tstr_auc)
    tstr_acc_mean, tstr_acc_std = _mean_std(tstr_acc)

    out = {
        "tstr_auc_mean": tstr_auc_mean,
        "tstr_auc_std": tstr_auc_std,
        "tstr_acc_mean": tstr_acc_mean,
        "tstr_acc_std": tstr_acc_std,
        "n_folds": len(tstr_auc),
    }

    if do_trtr and len(trtr_auc) > 0:
        trtr_auc_mean, trtr_auc_std = _mean_std(trtr_auc)
        trtr_acc_mean, trtr_acc_std = _mean_std(trtr_acc)
        out.update(
            {
                "trtr_auc_mean": trtr_auc_mean,
                "trtr_auc_std": trtr_auc_std,
                "trtr_acc_mean": trtr_acc_mean,
                "trtr_acc_std": trtr_acc_std,
            }
        )

    if do_c2st and len(c2st_auc) > 0:
        c2st_auc_mean, c2st_auc_std = _mean_std(c2st_auc)
        c2st_acc_mean, c2st_acc_std = _mean_std(c2st_acc)
        out.update(
            {
                "c2st_auc_mean": c2st_auc_mean,
                "c2st_auc_std": c2st_auc_std,
                "c2st_acc_mean": c2st_acc_mean,
                "c2st_acc_std": c2st_acc_std,
            }
        )

    return out


# ============================================================
# 4) CSV saving helpers
# ============================================================
def save_synth_csv(
    X_syn,
    y_syn=None,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_gobs_cuda0.csv",
    feature_names=None,
    target_col=TARGET_COL,
):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]
    assert len(feature_names) == m

    df = pd.DataFrame(X_syn, columns=feature_names)
    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        df.insert(0, target_col, y_syn)

    df.to_csv(out_csv, index=False)
    log(
        f"Saved CSV: {out_csv} | shape={df.shape} | "
        f"nan_frac={np.isnan(X_syn).mean():.6g}"
    )
    return out_csv


def save_synth_mask_csv(
    R_syn,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_mask_gobs_cuda0.csv",
):
    R_syn = np.asarray(R_syn).astype(int)
    dfR = pd.DataFrame(R_syn, columns=[f"r{j}" for j in range(R_syn.shape[1])])
    dfR.to_csv(out_csv, index=False)
    log(
        f"Saved mask CSV: {out_csv} | shape={dfR.shape} | "
        f"observed_frac={R_syn.mean():.6g}"
    )
    return out_csv


# ============================================================
# 5) BSTabDiff Optuna tuning
# ============================================================
def _sample_bstabdiff_params(trial):
    # Freeze the previously tuned BSTabDiff hyperparameters.
    params = dict(FIXED_BSTABDIFF_PARAMS)
    params["use_gobs"] = FIXED_USE_GOBS
    params["permute_features"] = FIXED_PERMUTE_FEATURES

    # Tune GO-BS parameters only.
    params["gobs_num_clusters"] = trial.suggest_int("gobs_num_clusters", 3, 10)
    params["gobs_metric"] = trial.suggest_categorical(
        "gobs_metric",
        ["correlation", "cosine", "euclidean", "manhattan"],
    )
    params["gobs_bins"] = trial.suggest_categorical("gobs_bins", [16, 32, 64])
    params["gobs_top_k"] = trial.suggest_categorical(
        "gobs_top_k", [None, 8, 16, 32, 64]
    )
    params["gobs_refine_order"] = trial.suggest_categorical(
        "gobs_refine_order", [True, False]
    )
    params["gobs_direction_select"] = trial.suggest_categorical(
        "gobs_direction_select", [True, False]
    )
    params["gobs_refine_passes"] = trial.suggest_int(
        "gobs_refine_passes", 0, 3
    )
    params["gobs_boundary_refine_passes"] = trial.suggest_int(
        "gobs_boundary_refine_passes", 1, 8
    )
    params["gobs_boundary_window"] = trial.suggest_categorical(
        "gobs_boundary_window", [4, 8, 12, 16]
    )
    params["gobs_lambda_cross"] = trial.suggest_float(
        "gobs_lambda_cross", 0.1, 5.0, log=True
    )
    params["gobs_gamma_balance"] = trial.suggest_float(
        "gobs_gamma_balance", 1e-6, 1e-1, log=True
    )

    return params


class _FrozenTrialAdapter:
    def __init__(self, params_dict):
        self.params_dict = params_dict

    def suggest_categorical(self, name, choices):
        if name in self.params_dict:
            return self.params_dict[name]
        return choices[0]

    def suggest_int(self, name, low, high, step=1, log=False):
        if name in self.params_dict:
            return int(self.params_dict[name])
        return low

    def suggest_float(self, name, low, high, step=None, log=False):
        if name in self.params_dict:
            return float(self.params_dict[name])
        return low


def _fit_bstabdiff_with_params(
    Xtr,
    ytr,
    params,
    device=REQUESTED_TORCH_DEVICE_STR,
    seed=0,
    verbose_every=0,
):
    feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(Xtr.shape[1])]

    gen = fit_block_subunit_generator(
        X=Xtr,
        feature_specs=feature_specs,
        y=ytr,
        M=params["M"],
        blocks=None,
        permute_features=False,
        prior_type=params["prior_type"],
        device=device,
        seed=seed,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=verbose_every,
        save_dir=None,
        save_name="bstabdiff_MiniBooNE_gobs_cuda0",
        save_best=True,
        use_ema=params["use_ema"],
        ema_decay=params["ema_decay"],
        return_train_info=False,
        use_gobs=params["use_gobs"],
        gobs_num_clusters=params["gobs_num_clusters"],
        gobs_metric=params["gobs_metric"],
        gobs_bins=params["gobs_bins"],
        gobs_top_k=params["gobs_top_k"],
        gobs_refine_order=params["gobs_refine_order"],
        gobs_direction_select=params["gobs_direction_select"],
        gobs_refine_passes=params["gobs_refine_passes"],
        gobs_boundary_refine_passes=params["gobs_boundary_refine_passes"],
        gobs_boundary_window=params["gobs_boundary_window"],
        gobs_lambda_cross=params["gobs_lambda_cross"],
        gobs_gamma_balance=params["gobs_gamma_balance"],
        gobs_use_cpu_kmeans=False,
    )
    return gen


def tune_bstabdiff_once(
    X,
    y,
    device=REQUESTED_TORCH_DEVICE_STR,
    n_trials=OPTUNA_TRIALS,
    seed=GLOBAL_SEED,
    inner_val_size=0.2,
    show_progress_bar=True,
):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).astype(int)

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=inner_val_size,
        random_state=seed,
    )
    (itr, iva), = splitter.split(X, y)
    X_inner_tr, y_inner_tr = X[itr], y[itr]
    X_inner_va, y_inner_va = X[iva], y[iva]

    def objective(trial):
        params = _sample_bstabdiff_params(trial)
        trial_seed = int(seed + 1000 + trial.number)

        try:
            gen_trial = _fit_bstabdiff_with_params(
                Xtr=X_inner_tr,
                ytr=y_inner_tr,
                params=params,
                device=device,
                seed=trial_seed,
                verbose_every=0,
            )

            X_syn, R_syn, y_syn = gen_trial.sample(n=int(params["n_syn_inner"]))
            X_syn = np.nan_to_num(np.asarray(X_syn, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_syn = np.asarray(y_syn).astype(int)

            if len(np.unique(y_syn)) < 2:
                return 0.0

            proxy_clf = make_clf(seed=trial_seed)
            proxy_clf.fit(X_syn, y_syn)
            score = _safe_binary_auc_or_acc(
                proxy_clf,
                X_inner_va,
                y_inner_va,
            )

            score_adj = float(score) - 1e-4 * float(params["M"])
            return score_adj

        except Exception:
            return 0.0

        finally:
            cleanup_torch()

    sampler = TPESampler(
        seed=seed,
        multivariate=True,
        warn_independent_sampling=False,
    )
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=OPTUNA_STUDY_NAME,
        storage=OPTUNA_STORAGE,
        load_if_exists=False,
    )

    with suppress_stdout_stderr():
        study.optimize(objective, n_trials=n_trials, show_progress_bar=show_progress_bar)

    best_full = _sample_bstabdiff_params(_FrozenTrialAdapter(dict(study.best_trial.params)))
    best_full["permute_features"] = FIXED_PERMUTE_FEATURES
    best_full["use_gobs"] = FIXED_USE_GOBS

    trials_df, top_trials_df, best_progression_df = print_optuna_summary_tables(study, top_k=10)
    return best_full, study, trials_df, top_trials_df, best_progression_df


def make_generator_fit_fn_from_best_params(
    best_params,
    device=REQUESTED_TORCH_DEVICE_STR,
    verbose_every=0,
):
    def _fit(Xtr, ytr, seed=0):
        gen = _fit_bstabdiff_with_params(
            Xtr=np.asarray(Xtr, dtype=np.float32),
            ytr=np.asarray(ytr).astype(int),
            params=best_params,
            device=device,
            seed=seed,
            verbose_every=verbose_every,
        )
        return gen
    return _fit


# ============================================================
# 6) Fidelity metric helpers
# ============================================================
def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return 0.0, 0.0

    values = np.concatenate([x, y])
    labels = np.concatenate([np.zeros(nx, dtype=int), np.ones(ny, dtype=int)])
    order = np.argsort(values)
    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1
    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny
    diff = cdf1 - cdf2

    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _rank_transform(X):
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    ranks = np.empty_like(X, dtype=float)
    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=float)
    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centered = X - mean
    var = (centered**2).mean(axis=0)
    eps = 1e-8
    std = np.sqrt(var + eps)
    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)
    return mean, var, skew, kurt


def compute_fidelity_metrics(
    X_real,
    y_real,
    X_syn,
    y_syn,
    feature_names=None,
    generator=None,
    random_state=0,
):
    X_real = np.asarray(X_real, dtype=float)
    X_syn = np.asarray(X_syn, dtype=float)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."
    n_features = X_real.shape[1]

    ks_stats = np.zeros(n_features, dtype=float)
    w1_stats = np.zeros(n_features, dtype=float)
    for j in range(n_features):
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    pearson_real = np.corrcoef(X_real, rowvar=False)
    pearson_syn = np.corrcoef(X_syn, rowvar=False)
    mask_offdiag = ~np.eye(n_features, dtype=bool)
    pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]
    pearson_mean = float(np.mean(pearson_abs_diff))
    pearson_max = float(np.max(pearson_abs_diff))

    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)
    spearman_real = np.corrcoef(ranks_real, rowvar=False)
    spearman_syn = np.corrcoef(ranks_syn, rowvar=False)
    spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]
    spearman_mean = float(np.mean(spearman_abs_diff))
    spearman_max = float(np.max(spearman_abs_diff))

    mi_real = mi_syn = mi_abs = None
    mi_mean = mi_max = None
    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)
        mi_real = mutual_info_classif(X_real, y_real, random_state=random_state)
        mi_syn = mutual_info_classif(X_syn, y_syn, random_state=random_state)
        mi_abs = np.abs(mi_real - mi_syn)
        mi_mean = float(np.mean(mi_abs))
        mi_max = float(np.max(mi_abs))

    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)
    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    ll_real = ll_syn = None
    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    nn_stats = {}
    try:
        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(X_real)
        dists, idx = nn.kneighbors(X_syn, return_distance=True)
        dists = dists.ravel()
        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_real": mi_real,
                "mi_syn": mi_syn,
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )
    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# 7) Load MiniBooNE dataset
# ============================================================
def load_miniboone_dataset(
    csv_path=CSV_PATH,
    label_col=TARGET_COL,
    expected_num_classes=EXPECTED_NUM_CLASSES,
):
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. "
            f"Columns: {list(df.columns)[:10]} ..."
        )

    y_raw = df[label_col].to_numpy()
    X_df = df.drop(columns=[label_col])
    X_df = X_df.select_dtypes(include=[np.number])

    if X_df.shape[1] == 0:
        raise ValueError("No numeric feature columns were found in the dataset.")

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Encode arbitrary binary labels as 0 and 1.
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, class_names = pd.factorize(y_raw, sort=True)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, class_names = pd.factorize(y_raw, sort=True)
        else:
            y = y_raw.astype(int, copy=False)
            class_names = unique

    y = np.asarray(y, dtype=int)
    classes, counts = np.unique(y, return_counts=True)

    if expected_num_classes is not None and len(classes) != expected_num_classes:
        raise ValueError(
            f"Expected {expected_num_classes} classes, but found {len(classes)}: "
            f"{classes.tolist()}"
        )

    if counts.min() < 5:
        raise ValueError(
            "The requested 5-fold stratified CV requires at least 5 samples "
            f"per class. Class counts: {dict(zip(classes.tolist(), counts.tolist()))}"
        )

    return X, y, list(X_df.columns), np.asarray(class_names)


# ============================================================
# 8) Main
# ============================================================
if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    log(f"Using torch device: {DEFAULT_TORCH_DEVICE}")
    log(f"Fixed permute_features: {FIXED_PERMUTE_FEATURES}")
    log(f"Fixed use_gobs: {FIXED_USE_GOBS}")
    log(f"Fixed BSTabDiff parameters: {FIXED_BSTABDIFF_PARAMS}")
    log("Optuna search space: GO-BS parameters only")
    log("Classifier: Logistic Regression only")

    # -------- Load MiniBooNE dataset --------
    X, y, feature_names, original_class_names = load_miniboone_dataset(
        csv_path=CSV_PATH,
        label_col=TARGET_COL,
        expected_num_classes=EXPECTED_NUM_CLASSES,
    )
    class_counts = dict(zip(*np.unique(y, return_counts=True)))
    log(
        f"Loaded {DATASET_NAME}: X.shape={X.shape}, y.shape={y.shape}, "
        f"encoded_classes={np.unique(y).tolist()}, class_counts={class_counts}"
    )
    log(f"Original class values: {original_class_names.tolist()}")

    # ============================================================
    # (0) ONE-TIME GO-BS-only Optuna tuning on full MiniBooNE
    # ============================================================
    log(
        f"\n=== Running ONE-TIME Optuna tuning for GO-BS "
        f"({OPTUNA_TRIALS} trials, GO-BS only, fixed BSTabDiff parameters, cuda:0) "
        f"on full {DATASET_NAME} ==="
    )
    t_tune0 = time.time()

    best_bstabdiff_params, study, trials_df, top_trials_df, best_progression_df = tune_bstabdiff_once(
        X=X,
        y=y,
        device=REQUESTED_TORCH_DEVICE_STR,
        n_trials=OPTUNA_TRIALS,
        seed=GLOBAL_SEED,
        inner_val_size=0.2,
        show_progress_bar=True,
    )

    t_tune = time.time() - t_tune0

    log("\n=== Best combined parameters after GO-BS-only Optuna study ===")
    log(str(best_bstabdiff_params))
    log(f"Best Optuna objective value: {study.best_value:.6f}")
    log(f"Tuning time: {t_tune:.2f} sec")

    # Save trial-wise summaries.
    trials_csv = "bstabdiff_MiniBooNE_gobs_optuna_trials_cuda0.csv"
    top_trials_csv = "bstabdiff_MiniBooNE_gobs_optuna_top_trials_cuda0.csv"
    progression_csv = "bstabdiff_MiniBooNE_gobs_optuna_best_progression_cuda0.csv"

    trials_df.to_csv(trials_csv, index=False)
    top_trials_df.to_csv(top_trials_csv, index=False)
    best_progression_df.to_csv(progression_csv, index=False)

    log("\nSaved Optuna summaries:")
    log(f"  - {trials_csv}")
    log(f"  - {top_trials_csv}")
    log(f"  - {progression_csv}")

    generator_fit_fn = make_generator_fit_fn_from_best_params(
        best_params=best_bstabdiff_params,
        device=REQUESTED_TORCH_DEVICE_STR,
        verbose_every=0,
    )

    # ============================================================
    # (A) Fit BSTabDiff once on full MiniBooNE data for reporting
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass

    cpu_before = proc.memory_info().rss if proc is not None else 0
    t0 = time.time()
    gen_full = generator_fit_fn(X, y, seed=GLOBAL_SEED)
    gen_train_time = time.time() - t0
    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)

    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated(device=DEFAULT_TORCH_DEVICE)
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024**3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024**3) if cpu_peak > 0 else 0.0

    log(f"\n=== BSTabDiff generator: resource usage (trained once on full {DATASET_NAME}) ===")
    log(f"  Train time: {gen_train_time:.2f} sec")
    log(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    log(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    # -------- Sample one synthetic dataset --------
    N_SYN = 200
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)

    synth_csv = "MiniBooNE_synth_BSTabDiff_n200_gobs_cuda0.csv"
    mask_csv = "MiniBooNE_synth_BSTabDiff_n200_mask_gobs_cuda0.csv"

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv=synth_csv,
        feature_names=feature_names,
        target_col=TARGET_COL,
    )
    save_synth_mask_csv(
        R_syn_global,
        out_csv=mask_csv,
    )

    # ============================================================
    # (B) 5x5 CV: strict fold-wise BSTabDiff + Logistic Regression
    # ============================================================
    log(
        "\n=== Evaluating STRICT BSTabDiff synthetic + Logistic Regression "
        "(5x5 TSTR/TRTR/C2ST) ==="
    )

    res = eval_5x5_tstr_foldwise_gen(
        X_real=X,
        y_real=y,
        generator_fit_fn=generator_fit_fn,
        n_syn=N_SYN,
        do_trtr=True,
        do_c2st=True,
        seed=0,
    )

    log(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
    log(
        f"  TSTR AUC: "
        f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}"
    )
    log(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
    log(
        f"  TRTR AUC: "
        f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}"
    )
    log(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
    log(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

    summary_df = pd.DataFrame(
        [
            {
                "Classifier": "Logistic Regression",
                "TSTR ACC (mean ± std)": (
                    f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}"
                ),
                "TSTR AUC (mean ± std)": (
                    f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}"
                ),
                "TRTR ACC (mean ± std)": (
                    f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}"
                ),
                "TRTR AUC (mean ± std)": (
                    f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}"
                ),
                "C2ST ACC (mean ± std)": (
                    f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}"
                ),
                "C2ST AUC (mean ± std)": (
                    f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}"
                ),
            }
        ]
    )

    summary_csv = "bstabdiff_MiniBooNE_cv_summary_lr_gobs_cuda0.csv"
    summary_df.to_csv(summary_csv, index=False)

    log(
        f"\n=== {DATASET_NAME}: STRICT BSTabDiff synthetic + "
        "Logistic Regression: 5x5 TSTR/TRTR/C2ST summary ==="
    )
    log(summary_df.to_markdown(index=False))
    log(f"Saved CV summary: {summary_csv}")

    # ============================================================
    # (C) Fidelity diagnostics
    # ============================================================
    fidelity = compute_fidelity_metrics(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        feature_names=feature_names,
        generator=gen_full,
        random_state=0,
    )

    log(f"\n=== {DATASET_NAME}: Fidelity metrics (real vs synthetic) ===")
    ks_mean = float(np.mean(fidelity["ks_per_feature"]))
    ks_max = float(np.max(fidelity["ks_per_feature"]))
    w1_mean = float(np.mean(fidelity["wasserstein_per_feature"]))
    w1_max = float(np.max(fidelity["wasserstein_per_feature"]))
    log(f"  KS statistic per feature:    mean={ks_mean:.4f}, max={ks_max:.4f}")
    log(f"  Wasserstein-1 per feature:   mean={w1_mean:.4f}, max={w1_max:.4f}")
    log(
        f"  Pearson |corr_real - corr_syn|: "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )
    log(
        f"  Spearman |corr_real - corr_syn|: "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]
    log("  Higher-order moments (mean abs diff over features):")
    log(f"    mean:  {higher['mean_abs_diff']:.4e}")
    log(f"    var:   {higher['var_abs_diff']:.4e}")
    log(f"    skew:  {higher['skew_abs_diff']:.4e}")
    log(f"    kurt:  {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        log(
            f"  Mutual information (feature-label) |ΔMI|: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    if "loglik_real_mean" in fidelity:
        log(
            f"  Log-likelihood (if available): "
            f"real={fidelity['loglik_real_mean']}, "
            f"synthetic={fidelity['loglik_syn_mean']}"
        )

    log(
        f"  NaN fraction (real): {fidelity['nan_frac_real']:.4e}, "
        f"NaN fraction (synthetic): {fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]
        log("  Privacy (NN distance from synthetic to nearest real):")
        log(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

Using torch device: cuda:0
Fixed permute_features: False
Fixed use_gobs: True
Fixed BSTabDiff parameters: {'M': 64, 'prior_type': 'diffusion', 'prior_epochs': 800, 'prior_batch': 64, 'prior_lr': 0.000921878160004023, 'use_ema': True, 'ema_decay': 0.9908430906937514, 'n_syn_inner': 256}
Optuna search space: GO-BS parameters only
Classifier: Logistic Regression only
Loaded MiniBooNE: X.shape=(130064, 50), y.shape=(130064,), encoded_classes=[0, 1], class_counts={np.int64(0): np.int64(93565), np.int64(1): np.int64(36499)}
Original class values: [0, 1]

=== Running ONE-TIME Optuna tuning for GO-BS (150 trials, GO-BS only, fixed BSTabDiff parameters, cuda:0) on full MiniBooNE ===


  0%|          | 0/150 [00:00<?, ?it/s]


=== Optuna trial summary ===
Total trials: 150 | COMPLETE: 150 | PRUNED: 0 | FAIL: 0

=== Top 10 Optuna trials by objective ===
|   trial | state    |    value |   best_so_far |   duration_sec | datetime_start             | datetime_complete          |   gobs_num_clusters | gobs_metric   |   gobs_bins |   gobs_top_k | gobs_refine_order   | gobs_direction_select   |   gobs_refine_passes |   gobs_boundary_refine_passes |   gobs_boundary_window |   gobs_lambda_cross |   gobs_gamma_balance |
|--------:|:---------|---------:|--------------:|---------------:|:---------------------------|:---------------------------|--------------------:|:--------------|------------:|-------------:|:--------------------|:------------------------|---------------------:|------------------------------:|-----------------------:|--------------------:|---------------------:|
|     125 | COMPLETE | 0.939449 |      0.939449 |       11.6271  | 2026-07-25 21:43:31.927469 | 2026-07-25 21:43:43.554532 |                 

# BSTabDiff || MiniBooNE || LDHSS || GO-BS-FC || Tuning

In [ ]:
import os

# Suppress Python warnings before importing third-party packages.
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

import gc
import time
import random
import contextlib
import sys

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# PyTorch is still required by BSTabDiff and for CUDA resource reporting.
try:
    import torch
except Exception:
    torch = None

# BSTabDiff
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Global runtime configuration
# ============================================================
GLOBAL_SEED = 42

DATASET_NAME = "MiniBooNE"
CSV_PATH = "MiniBooNE_labeled.csv"
TARGET_COL = "label"
EXPECTED_NUM_CLASSES = 2

# Fixed to CUDA:0 for BSTabDiff generator training
REQUESTED_TORCH_DEVICE_STR = "cuda:0"

# Tune GO-BS-FC only; all previously optimized BSTabDiff settings are fixed.
OPTUNA_TRIALS = 150
OPTUNA_STUDY_NAME = "bstabdiff_MiniBooNE_gobsfc_cuda0_optuna150"
OPTUNA_STORAGE = None

FIXED_PERMUTE_FEATURES = False
FIXED_USE_GOBS = True
FIXED_USE_GOBS_FC = True

# Best MiniBooNE BSTabDiff settings from the completed no-GO-BS study.
# These remain fixed in every one of the 150 GO-BS-FC trials.
FIXED_BSTABDIFF_PARAMS = {
    "M": 64,
    "prior_type": "diffusion",
    "prior_epochs": 800,
    "prior_batch": 64,
    "prior_lr": 0.000921878160004023,
    "use_ema": True,
    "ema_decay": 0.9908430906937514,
    "n_syn_inner": 256,
}

optuna.logging.set_verbosity(optuna.logging.ERROR)


# ============================================================
# Small console helper
# ============================================================
def log(msg):
    print(msg, flush=True)


# ============================================================
# Optuna output suppression + summaries
# ============================================================
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def _trial_duration_seconds(trial):
    if trial.datetime_start is None or trial.datetime_complete is None:
        return np.nan
    return (trial.datetime_complete - trial.datetime_start).total_seconds()


def build_optuna_trial_dataframe(study):
    rows = []
    best_so_far = -np.inf

    for t in study.trials:
        value = t.value if t.value is not None else np.nan
        if t.state.name == "COMPLETE" and np.isfinite(value):
            best_so_far = max(best_so_far, value)

        row = {
            "trial": t.number,
            "state": t.state.name,
            "value": value,
            "best_so_far": best_so_far if np.isfinite(best_so_far) else np.nan,
            "duration_sec": _trial_duration_seconds(t),
            "datetime_start": str(t.datetime_start) if t.datetime_start is not None else None,
            "datetime_complete": str(t.datetime_complete) if t.datetime_complete is not None else None,
        }

        for k, v in t.params.items():
            row[f"param__{k}"] = v

        rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    for col in ["value", "best_so_far", "duration_sec"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def get_top_trials_df(trials_df, top_k=10):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    df = df.sort_values(["value", "trial"], ascending=[False, True]).reset_index(drop=True)
    return df.head(top_k)


def get_best_so_far_progression_df(trials_df):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    best_rows = []
    running_best = -np.inf
    for _, row in df.sort_values("trial").iterrows():
        if row["value"] > running_best:
            running_best = row["value"]
            best_rows.append(row)

    if len(best_rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(best_rows).reset_index(drop=True)


def pretty_param_columns(df):
    cols = [c for c in df.columns if c.startswith("param__")]
    return df.rename(columns={c: c.replace("param__", "") for c in cols})


def print_optuna_summary_tables(study, top_k=10):
    trials_df = build_optuna_trial_dataframe(study)
    top_df = get_top_trials_df(trials_df, top_k=top_k)
    best_prog_df = get_best_so_far_progression_df(trials_df)

    log("\n=== Optuna trial summary ===")
    if len(trials_df) == 0:
        log("No trials found.")
        return trials_df, top_df, best_prog_df

    n_complete = int((trials_df["state"] == "COMPLETE").sum())
    n_pruned = int((trials_df["state"] == "PRUNED").sum())
    n_fail = int((trials_df["state"] == "FAIL").sum())

    log(f"Total trials: {len(trials_df)} | COMPLETE: {n_complete} | PRUNED: {n_pruned} | FAIL: {n_fail}")

    if len(top_df) > 0:
        top_show = pretty_param_columns(top_df.copy())
        log(f"\n=== Top {min(top_k, len(top_show))} Optuna trials by objective ===")
        log(top_show.to_markdown(index=False))

    if len(best_prog_df) > 0:
        prog_show = pretty_param_columns(best_prog_df.copy())
        log("\n=== Best-so-far progression ===")
        log(prog_show.to_markdown(index=False))

    return trials_df, top_df, best_prog_df


# ============================================================
# Device / seed helpers
# ============================================================
def resolve_torch_device(device_str: str = REQUESTED_TORCH_DEVICE_STR):
    if torch is None:
        return None
    if not torch.cuda.is_available():
        return torch.device("cpu")
    if not device_str.startswith("cuda:"):
        return torch.device(device_str)
    idx = int(device_str.split(":")[1])
    ndev = torch.cuda.device_count()
    if idx >= ndev:
        raise RuntimeError(
            f"Requested {device_str}, but only {ndev} CUDA device(s) are visible."
        )
    return torch.device(device_str)


if torch is not None:
    DEFAULT_TORCH_DEVICE = resolve_torch_device(REQUESTED_TORCH_DEVICE_STR)
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
else:
    DEFAULT_TORCH_DEVICE = None


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cleanup_torch():
    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    gc.collect()


# ============================================================
# 1) Logistic Regression classifier
# ============================================================
def make_clf(seed=0):
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    n_jobs=None,
                    random_state=seed,
                ),
            ),
        ]
    )


# ============================================================
# 2) Classification metric helpers
# ============================================================
def _predict_proba_binary(clf, X):
    if hasattr(clf, "predict_proba"):
        proba = np.asarray(clf.predict_proba(X), dtype=float)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(clf, "decision_function"):
        scores = np.asarray(clf.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-scores))

    raise ValueError("Classifier has neither predict_proba nor decision_function.")


def _safe_binary_auc_or_acc(clf, X_val, y_val):
    y_val = np.asarray(y_val).astype(int)

    if len(np.unique(y_val)) < 2:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))

    try:
        proba = _predict_proba_binary(clf, X_val)
        return float(roc_auc_score(y_val, proba))
    except Exception:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))


# ============================================================
# 3) Strict 5x5 CV with fold-wise generator training
# ============================================================
def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    generator_fit_fn=None,
    n_syn=200,
    do_trtr=True,
    do_c2st=True,
    seed=0,
):
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided.")

    X_real = np.asarray(X_real)
    y_real = np.asarray(y_real).astype(int)

    classes, counts = np.unique(y_real, return_counts=True)
    if len(classes) != 2:
        raise ValueError(
            f"Binary classification requires exactly 2 classes, but found "
            f"{len(classes)}: {classes.tolist()}"
        )
    if counts.min() < 5:
        raise ValueError(
            "RepeatedStratifiedKFold with n_splits=5 requires at least "
            f"5 samples per class. Class counts: "
            f"{dict(zip(classes.tolist(), counts.tolist()))}"
        )

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)
        X_syn_fold = np.nan_to_num(
            np.asarray(X_syn_fold, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_syn_fold = np.asarray(y_syn_fold).astype(int)

        if len(np.unique(y_syn_fold)) < 2:
            raise RuntimeError(
                f"Fold {fold_id}: the synthetic dataset contains fewer than two classes."
            )

        # TSTR: train on synthetic, test on real.
        clf_tstr = make_clf(seed=fold_seed)
        clf_tstr.fit(X_syn_fold, y_syn_fold)
        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)
        tstr_auc.append(float(roc_auc_score(yte, proba_t)))
        tstr_acc.append(float(accuracy_score(yte, pred_t)))

        # TRTR: train on real, test on real.
        if do_trtr:
            clf_trtr = make_clf(seed=fold_seed)
            clf_trtr.fit(Xtr, ytr)
            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)
            trtr_auc.append(float(roc_auc_score(yte, proba_r)))
            trtr_acc.append(float(accuracy_score(yte, pred_r)))

        # C2ST: distinguish real=1 from synthetic=0.
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)]
            )
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf(seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)
            c2st_auc.append(float(roc_auc_score(ymix_te, proba_m)))
            c2st_acc.append(float(accuracy_score(ymix_te, pred_m)))

        log(
            f"[Fold {fold_id:02d}] "
            f"TSTR ACC={tstr_acc[-1]:.4f}, "
            f"AUC={tstr_auc[-1]:.4f}"
        )
        cleanup_torch()

    def _mean_std(values):
        values = np.asarray(values, dtype=float)
        return float(np.mean(values)), float(np.std(values))

    tstr_auc_mean, tstr_auc_std = _mean_std(tstr_auc)
    tstr_acc_mean, tstr_acc_std = _mean_std(tstr_acc)

    out = {
        "tstr_auc_mean": tstr_auc_mean,
        "tstr_auc_std": tstr_auc_std,
        "tstr_acc_mean": tstr_acc_mean,
        "tstr_acc_std": tstr_acc_std,
        "n_folds": len(tstr_auc),
    }

    if do_trtr and len(trtr_auc) > 0:
        trtr_auc_mean, trtr_auc_std = _mean_std(trtr_auc)
        trtr_acc_mean, trtr_acc_std = _mean_std(trtr_acc)
        out.update(
            {
                "trtr_auc_mean": trtr_auc_mean,
                "trtr_auc_std": trtr_auc_std,
                "trtr_acc_mean": trtr_acc_mean,
                "trtr_acc_std": trtr_acc_std,
            }
        )

    if do_c2st and len(c2st_auc) > 0:
        c2st_auc_mean, c2st_auc_std = _mean_std(c2st_auc)
        c2st_acc_mean, c2st_acc_std = _mean_std(c2st_acc)
        out.update(
            {
                "c2st_auc_mean": c2st_auc_mean,
                "c2st_auc_std": c2st_auc_std,
                "c2st_acc_mean": c2st_acc_mean,
                "c2st_acc_std": c2st_acc_std,
            }
        )

    return out


# ============================================================
# 4) CSV saving helpers
# ============================================================
def save_synth_csv(
    X_syn,
    y_syn=None,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_gobsfc_cuda0.csv",
    feature_names=None,
    target_col=TARGET_COL,
):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]
    assert len(feature_names) == m

    df = pd.DataFrame(X_syn, columns=feature_names)
    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        df.insert(0, target_col, y_syn)

    df.to_csv(out_csv, index=False)
    log(
        f"Saved CSV: {out_csv} | shape={df.shape} | "
        f"nan_frac={np.isnan(X_syn).mean():.6g}"
    )
    return out_csv


def save_synth_mask_csv(
    R_syn,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_mask_gobsfc_cuda0.csv",
):
    R_syn = np.asarray(R_syn).astype(int)
    dfR = pd.DataFrame(R_syn, columns=[f"r{j}" for j in range(R_syn.shape[1])])
    dfR.to_csv(out_csv, index=False)
    log(
        f"Saved mask CSV: {out_csv} | shape={dfR.shape} | "
        f"observed_frac={R_syn.mean():.6g}"
    )
    return out_csv


# ============================================================
# 5) BSTabDiff Optuna tuning
# ============================================================
def _sample_bstabdiff_params(trial):
    # Freeze the previously tuned BSTabDiff hyperparameters.
    params = dict(FIXED_BSTABDIFF_PARAMS)
    params["use_gobs"] = FIXED_USE_GOBS
    params["use_gobs_fc"] = FIXED_USE_GOBS_FC
    params["permute_features"] = FIXED_PERMUTE_FEATURES

    # Tune GO-BS-FC parameters only.
    params["gobs_num_clusters"] = trial.suggest_int("gobs_num_clusters", 3, 10)
    params["gobs_metric"] = trial.suggest_categorical(
        "gobs_metric",
        ["correlation", "cosine", "euclidean", "manhattan"],
    )
    params["gobs_bins"] = trial.suggest_categorical("gobs_bins", [16, 32, 64])
    params["gobs_top_k"] = trial.suggest_categorical(
        "gobs_top_k", [None, 8, 16, 32, 64]
    )
    params["gobs_refine_order"] = trial.suggest_categorical(
        "gobs_refine_order", [True, False]
    )
    params["gobs_direction_select"] = trial.suggest_categorical(
        "gobs_direction_select", [True, False]
    )
    params["gobs_refine_passes"] = trial.suggest_int(
        "gobs_refine_passes", 0, 3
    )
    params["gobs_boundary_refine_passes"] = trial.suggest_int(
        "gobs_boundary_refine_passes", 1, 8
    )
    params["gobs_boundary_window"] = trial.suggest_categorical(
        "gobs_boundary_window", [4, 8, 12, 16]
    )
    params["gobs_lambda_cross"] = trial.suggest_float(
        "gobs_lambda_cross", 0.1, 5.0, log=True
    )
    params["gobs_gamma_balance"] = trial.suggest_float(
        "gobs_gamma_balance", 1e-6, 1e-1, log=True
    )

    return params


class _FrozenTrialAdapter:
    def __init__(self, params_dict):
        self.params_dict = params_dict

    def suggest_categorical(self, name, choices):
        if name in self.params_dict:
            return self.params_dict[name]
        return choices[0]

    def suggest_int(self, name, low, high, step=1, log=False):
        if name in self.params_dict:
            return int(self.params_dict[name])
        return low

    def suggest_float(self, name, low, high, step=None, log=False):
        if name in self.params_dict:
            return float(self.params_dict[name])
        return low


def _fit_bstabdiff_with_params(
    Xtr,
    ytr,
    params,
    device=REQUESTED_TORCH_DEVICE_STR,
    seed=0,
    verbose_every=0,
):
    feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(Xtr.shape[1])]

    gen = fit_block_subunit_generator(
        X=Xtr,
        feature_specs=feature_specs,
        y=ytr,
        M=params["M"],
        blocks=None,
        permute_features=False,
        prior_type=params["prior_type"],
        device=device,
        seed=seed,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=verbose_every,
        save_dir=None,
        save_name="bstabdiff_MiniBooNE_gobsfc_cuda0",
        save_best=True,
        use_ema=params["use_ema"],
        ema_decay=params["ema_decay"],
        return_train_info=False,
        use_gobs=params["use_gobs"],
        use_gobs_fc=params["use_gobs_fc"],
        gobs_num_clusters=params["gobs_num_clusters"],
        gobs_metric=params["gobs_metric"],
        gobs_bins=params["gobs_bins"],
        gobs_top_k=params["gobs_top_k"],
        gobs_refine_order=params["gobs_refine_order"],
        gobs_direction_select=params["gobs_direction_select"],
        gobs_refine_passes=params["gobs_refine_passes"],
        gobs_boundary_refine_passes=params["gobs_boundary_refine_passes"],
        gobs_boundary_window=params["gobs_boundary_window"],
        gobs_lambda_cross=params["gobs_lambda_cross"],
        gobs_gamma_balance=params["gobs_gamma_balance"],
        gobs_use_cpu_kmeans=False,
    )
    return gen


def tune_bstabdiff_once(
    X,
    y,
    device=REQUESTED_TORCH_DEVICE_STR,
    n_trials=OPTUNA_TRIALS,
    seed=GLOBAL_SEED,
    inner_val_size=0.2,
    show_progress_bar=True,
):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).astype(int)

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=inner_val_size,
        random_state=seed,
    )
    (itr, iva), = splitter.split(X, y)
    X_inner_tr, y_inner_tr = X[itr], y[itr]
    X_inner_va, y_inner_va = X[iva], y[iva]

    def objective(trial):
        params = _sample_bstabdiff_params(trial)
        trial_seed = int(seed + 1000 + trial.number)

        try:
            gen_trial = _fit_bstabdiff_with_params(
                Xtr=X_inner_tr,
                ytr=y_inner_tr,
                params=params,
                device=device,
                seed=trial_seed,
                verbose_every=0,
            )

            X_syn, R_syn, y_syn = gen_trial.sample(n=int(params["n_syn_inner"]))
            X_syn = np.nan_to_num(np.asarray(X_syn, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_syn = np.asarray(y_syn).astype(int)

            if len(np.unique(y_syn)) < 2:
                return 0.0

            proxy_clf = make_clf(seed=trial_seed)
            proxy_clf.fit(X_syn, y_syn)
            score = _safe_binary_auc_or_acc(
                proxy_clf,
                X_inner_va,
                y_inner_va,
            )

            score_adj = float(score) - 1e-4 * float(params["M"])
            return score_adj

        except Exception:
            return 0.0

        finally:
            cleanup_torch()

    sampler = TPESampler(
        seed=seed,
        multivariate=True,
        warn_independent_sampling=False,
    )
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=OPTUNA_STUDY_NAME,
        storage=OPTUNA_STORAGE,
        load_if_exists=False,
    )

    with suppress_stdout_stderr():
        study.optimize(objective, n_trials=n_trials, show_progress_bar=show_progress_bar)

    best_full = _sample_bstabdiff_params(_FrozenTrialAdapter(dict(study.best_trial.params)))
    best_full["permute_features"] = FIXED_PERMUTE_FEATURES
    best_full["use_gobs"] = FIXED_USE_GOBS
    best_full["use_gobs_fc"] = FIXED_USE_GOBS_FC

    trials_df, top_trials_df, best_progression_df = print_optuna_summary_tables(study, top_k=10)
    return best_full, study, trials_df, top_trials_df, best_progression_df


def make_generator_fit_fn_from_best_params(
    best_params,
    device=REQUESTED_TORCH_DEVICE_STR,
    verbose_every=0,
):
    def _fit(Xtr, ytr, seed=0):
        gen = _fit_bstabdiff_with_params(
            Xtr=np.asarray(Xtr, dtype=np.float32),
            ytr=np.asarray(ytr).astype(int),
            params=best_params,
            device=device,
            seed=seed,
            verbose_every=verbose_every,
        )
        return gen
    return _fit


# ============================================================
# 6) Fidelity metric helpers
# ============================================================
def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return 0.0, 0.0

    values = np.concatenate([x, y])
    labels = np.concatenate([np.zeros(nx, dtype=int), np.ones(ny, dtype=int)])
    order = np.argsort(values)
    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1
    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny
    diff = cdf1 - cdf2

    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _rank_transform(X):
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    ranks = np.empty_like(X, dtype=float)
    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=float)
    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centered = X - mean
    var = (centered**2).mean(axis=0)
    eps = 1e-8
    std = np.sqrt(var + eps)
    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)
    return mean, var, skew, kurt


def compute_fidelity_metrics(
    X_real,
    y_real,
    X_syn,
    y_syn,
    feature_names=None,
    generator=None,
    random_state=0,
):
    X_real = np.asarray(X_real, dtype=float)
    X_syn = np.asarray(X_syn, dtype=float)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."
    n_features = X_real.shape[1]

    ks_stats = np.zeros(n_features, dtype=float)
    w1_stats = np.zeros(n_features, dtype=float)
    for j in range(n_features):
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    pearson_real = np.corrcoef(X_real, rowvar=False)
    pearson_syn = np.corrcoef(X_syn, rowvar=False)
    mask_offdiag = ~np.eye(n_features, dtype=bool)
    pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]
    pearson_mean = float(np.mean(pearson_abs_diff))
    pearson_max = float(np.max(pearson_abs_diff))

    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)
    spearman_real = np.corrcoef(ranks_real, rowvar=False)
    spearman_syn = np.corrcoef(ranks_syn, rowvar=False)
    spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]
    spearman_mean = float(np.mean(spearman_abs_diff))
    spearman_max = float(np.max(spearman_abs_diff))

    mi_real = mi_syn = mi_abs = None
    mi_mean = mi_max = None
    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)
        mi_real = mutual_info_classif(X_real, y_real, random_state=random_state)
        mi_syn = mutual_info_classif(X_syn, y_syn, random_state=random_state)
        mi_abs = np.abs(mi_real - mi_syn)
        mi_mean = float(np.mean(mi_abs))
        mi_max = float(np.max(mi_abs))

    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)
    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    ll_real = ll_syn = None
    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    nn_stats = {}
    try:
        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(X_real)
        dists, idx = nn.kneighbors(X_syn, return_distance=True)
        dists = dists.ravel()
        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_real": mi_real,
                "mi_syn": mi_syn,
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )
    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# 7) Load MiniBooNE dataset
# ============================================================
def load_miniboone_dataset(
    csv_path=CSV_PATH,
    label_col=TARGET_COL,
    expected_num_classes=EXPECTED_NUM_CLASSES,
):
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. "
            f"Columns: {list(df.columns)[:10]} ..."
        )

    y_raw = df[label_col].to_numpy()
    X_df = df.drop(columns=[label_col])
    X_df = X_df.select_dtypes(include=[np.number])

    if X_df.shape[1] == 0:
        raise ValueError("No numeric feature columns were found in the dataset.")

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Encode arbitrary binary labels as 0 and 1.
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, class_names = pd.factorize(y_raw, sort=True)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, class_names = pd.factorize(y_raw, sort=True)
        else:
            y = y_raw.astype(int, copy=False)
            class_names = unique

    y = np.asarray(y, dtype=int)
    classes, counts = np.unique(y, return_counts=True)

    if expected_num_classes is not None and len(classes) != expected_num_classes:
        raise ValueError(
            f"Expected {expected_num_classes} classes, but found {len(classes)}: "
            f"{classes.tolist()}"
        )

    if counts.min() < 5:
        raise ValueError(
            "The requested 5-fold stratified CV requires at least 5 samples "
            f"per class. Class counts: {dict(zip(classes.tolist(), counts.tolist()))}"
        )

    return X, y, list(X_df.columns), np.asarray(class_names)


# ============================================================
# 8) Main
# ============================================================
if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    log(f"Using torch device: {DEFAULT_TORCH_DEVICE}")
    log(f"Fixed permute_features: {FIXED_PERMUTE_FEATURES}")
    log(f"Fixed use_gobs: {FIXED_USE_GOBS}")
    log(f"Fixed use_gobs_fc: {FIXED_USE_GOBS_FC}")
    log(f"Fixed BSTabDiff parameters: {FIXED_BSTABDIFF_PARAMS}")
    log("Optuna search space: GO-BS-FC parameters only")
    log("Classifier: Logistic Regression only")

    # -------- Load MiniBooNE dataset --------
    X, y, feature_names, original_class_names = load_miniboone_dataset(
        csv_path=CSV_PATH,
        label_col=TARGET_COL,
        expected_num_classes=EXPECTED_NUM_CLASSES,
    )
    class_counts = dict(zip(*np.unique(y, return_counts=True)))
    log(
        f"Loaded {DATASET_NAME}: X.shape={X.shape}, y.shape={y.shape}, "
        f"encoded_classes={np.unique(y).tolist()}, class_counts={class_counts}"
    )
    log(f"Original class values: {original_class_names.tolist()}")

    # ============================================================
    # (0) ONE-TIME GO-BS-FC-only Optuna tuning on full MiniBooNE
    # ============================================================
    log(
        f"\n=== Running ONE-TIME Optuna tuning for GO-BS-FC "
        f"({OPTUNA_TRIALS} trials, GO-BS-FC only, fixed BSTabDiff parameters, cuda:0) "
        f"on full {DATASET_NAME} ==="
    )
    t_tune0 = time.time()

    best_bstabdiff_params, study, trials_df, top_trials_df, best_progression_df = tune_bstabdiff_once(
        X=X,
        y=y,
        device=REQUESTED_TORCH_DEVICE_STR,
        n_trials=OPTUNA_TRIALS,
        seed=GLOBAL_SEED,
        inner_val_size=0.2,
        show_progress_bar=True,
    )

    t_tune = time.time() - t_tune0

    log("\n=== Best combined parameters after GO-BS-FC-only Optuna study ===")
    log(str(best_bstabdiff_params))
    log(f"Best Optuna objective value: {study.best_value:.6f}")
    log(f"Tuning time: {t_tune:.2f} sec")

    # Save trial-wise summaries.
    trials_csv = "bstabdiff_MiniBooNE_gobsfc_optuna_trials_cuda0.csv"
    top_trials_csv = "bstabdiff_MiniBooNE_gobsfc_optuna_top_trials_cuda0.csv"
    progression_csv = "bstabdiff_MiniBooNE_gobsfc_optuna_best_progression_cuda0.csv"

    trials_df.to_csv(trials_csv, index=False)
    top_trials_df.to_csv(top_trials_csv, index=False)
    best_progression_df.to_csv(progression_csv, index=False)

    log("\nSaved Optuna summaries:")
    log(f"  - {trials_csv}")
    log(f"  - {top_trials_csv}")
    log(f"  - {progression_csv}")

    generator_fit_fn = make_generator_fit_fn_from_best_params(
        best_params=best_bstabdiff_params,
        device=REQUESTED_TORCH_DEVICE_STR,
        verbose_every=0,
    )

    # ============================================================
    # (A) Fit BSTabDiff once on full MiniBooNE data for reporting
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass

    cpu_before = proc.memory_info().rss if proc is not None else 0
    t0 = time.time()
    gen_full = generator_fit_fn(X, y, seed=GLOBAL_SEED)
    gen_train_time = time.time() - t0
    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)

    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated(device=DEFAULT_TORCH_DEVICE)
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024**3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024**3) if cpu_peak > 0 else 0.0

    log(f"\n=== BSTabDiff generator: resource usage (trained once on full {DATASET_NAME}) ===")
    log(f"  Train time: {gen_train_time:.2f} sec")
    log(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    log(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    # -------- Sample one synthetic dataset --------
    N_SYN = 200
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)

    synth_csv = "MiniBooNE_synth_BSTabDiff_n200_gobsfc_cuda0.csv"
    mask_csv = "MiniBooNE_synth_BSTabDiff_n200_mask_gobsfc_cuda0.csv"

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv=synth_csv,
        feature_names=feature_names,
        target_col=TARGET_COL,
    )
    save_synth_mask_csv(
        R_syn_global,
        out_csv=mask_csv,
    )

    # ============================================================
    # (B) 5x5 CV: strict fold-wise BSTabDiff + Logistic Regression
    # ============================================================
    log(
        "\n=== Evaluating STRICT BSTabDiff synthetic + Logistic Regression "
        "(5x5 TSTR/TRTR/C2ST) ==="
    )

    res = eval_5x5_tstr_foldwise_gen(
        X_real=X,
        y_real=y,
        generator_fit_fn=generator_fit_fn,
        n_syn=N_SYN,
        do_trtr=True,
        do_c2st=True,
        seed=0,
    )

    log(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
    log(
        f"  TSTR AUC: "
        f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}"
    )
    log(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
    log(
        f"  TRTR AUC: "
        f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}"
    )
    log(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
    log(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

    summary_df = pd.DataFrame(
        [
            {
                "Classifier": "Logistic Regression",
                "TSTR ACC (mean ± std)": (
                    f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}"
                ),
                "TSTR AUC (mean ± std)": (
                    f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}"
                ),
                "TRTR ACC (mean ± std)": (
                    f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}"
                ),
                "TRTR AUC (mean ± std)": (
                    f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}"
                ),
                "C2ST ACC (mean ± std)": (
                    f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}"
                ),
                "C2ST AUC (mean ± std)": (
                    f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}"
                ),
            }
        ]
    )

    summary_csv = "bstabdiff_MiniBooNE_cv_summary_lr_gobsfc_cuda0.csv"
    summary_df.to_csv(summary_csv, index=False)

    log(
        f"\n=== {DATASET_NAME}: STRICT BSTabDiff synthetic + "
        "Logistic Regression: 5x5 TSTR/TRTR/C2ST summary ==="
    )
    log(summary_df.to_markdown(index=False))
    log(f"Saved CV summary: {summary_csv}")

    # ============================================================
    # (C) Fidelity diagnostics
    # ============================================================
    fidelity = compute_fidelity_metrics(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        feature_names=feature_names,
        generator=gen_full,
        random_state=0,
    )

    log(f"\n=== {DATASET_NAME}: Fidelity metrics (real vs synthetic) ===")
    ks_mean = float(np.mean(fidelity["ks_per_feature"]))
    ks_max = float(np.max(fidelity["ks_per_feature"]))
    w1_mean = float(np.mean(fidelity["wasserstein_per_feature"]))
    w1_max = float(np.max(fidelity["wasserstein_per_feature"]))
    log(f"  KS statistic per feature:    mean={ks_mean:.4f}, max={ks_max:.4f}")
    log(f"  Wasserstein-1 per feature:   mean={w1_mean:.4f}, max={w1_max:.4f}")
    log(
        f"  Pearson |corr_real - corr_syn|: "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )
    log(
        f"  Spearman |corr_real - corr_syn|: "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]
    log("  Higher-order moments (mean abs diff over features):")
    log(f"    mean:  {higher['mean_abs_diff']:.4e}")
    log(f"    var:   {higher['var_abs_diff']:.4e}")
    log(f"    skew:  {higher['skew_abs_diff']:.4e}")
    log(f"    kurt:  {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        log(
            f"  Mutual information (feature-label) |ΔMI|: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    if "loglik_real_mean" in fidelity:
        log(
            f"  Log-likelihood (if available): "
            f"real={fidelity['loglik_real_mean']}, "
            f"synthetic={fidelity['loglik_syn_mean']}"
        )

    log(
        f"  NaN fraction (real): {fidelity['nan_frac_real']:.4e}, "
        f"NaN fraction (synthetic): {fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]
        log("  Privacy (NN distance from synthetic to nearest real):")
        log(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

Using torch device: cuda:0
Fixed permute_features: False
Fixed use_gobs: True
Fixed use_gobs_fc: True
Fixed BSTabDiff parameters: {'M': 64, 'prior_type': 'diffusion', 'prior_epochs': 800, 'prior_batch': 64, 'prior_lr': 0.000921878160004023, 'use_ema': True, 'ema_decay': 0.9908430906937514, 'n_syn_inner': 256}
Optuna search space: GO-BS-FC parameters only
Classifier: Logistic Regression only
Loaded MiniBooNE: X.shape=(130064, 50), y.shape=(130064,), encoded_classes=[0, 1], class_counts={np.int64(0): np.int64(93565), np.int64(1): np.int64(36499)}
Original class values: [0, 1]

=== Running ONE-TIME Optuna tuning for GO-BS-FC (150 trials, GO-BS-FC only, fixed BSTabDiff parameters, cuda:0) on full MiniBooNE ===


  0%|          | 0/150 [00:00<?, ?it/s]


=== Optuna trial summary ===
Total trials: 150 | COMPLETE: 150 | PRUNED: 0 | FAIL: 0

=== Top 10 Optuna trials by objective ===
|   trial | state    |    value |   best_so_far |   duration_sec | datetime_start             | datetime_complete          |   gobs_num_clusters | gobs_metric   |   gobs_bins |   gobs_top_k | gobs_refine_order   | gobs_direction_select   |   gobs_refine_passes |   gobs_boundary_refine_passes |   gobs_boundary_window |   gobs_lambda_cross |   gobs_gamma_balance |
|--------:|:---------|---------:|--------------:|---------------:|:---------------------------|:---------------------------|--------------------:|:--------------|------------:|-------------:|:--------------------|:------------------------|---------------------:|------------------------------:|-----------------------:|--------------------:|---------------------:|
|      13 | COMPLETE | 0.939477 |      0.939477 |       11.1311  | 2026-07-25 22:30:42.766074 | 2026-07-25 22:30:53.897136 |                 

# BSTabDiff || MiniBooNE || default

In [ ]:
import os

# Suppress Python warnings before importing third-party packages.
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

import gc
import time
import random
import contextlib
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors


# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# PyTorch is still required by BSTabDiff and for CUDA resource reporting.
try:
    import torch
except Exception:
    torch = None

# BSTabDiff
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Global runtime configuration
# ============================================================
GLOBAL_SEED = 42

DATASET_NAME = "MiniBooNE"
CSV_PATH = "MiniBooNE_labeled.csv"
TARGET_COL = "label"
EXPECTED_NUM_CLASSES = 2

# Fixed to CUDA:0 for BSTabDiff generator training.
REQUESTED_TORCH_DEVICE_STR = "cuda:0"

# Nominal dataset-independent default. It is capped at the observed
# feature count so that np.array_split never creates empty blocks.
DEFAULT_M_NOMINAL = 64

# Fixed BSTabDiff defaults: no tuning, no GO-BS/GO-BS-FC,
# no feature permutation, and no predefined blocks.
DEFAULT_PRIOR_TYPE = "diffusion"
DEFAULT_PRIOR_EPOCHS = 10_000
DEFAULT_PRIOR_BATCH = 128
DEFAULT_PRIOR_LR = 1e-3
DEFAULT_USE_EMA = True
DEFAULT_EMA_DECAY = 0.999

FIXED_PERMUTE_FEATURES = False
FIXED_USE_GOBS = False
FIXED_USE_GOBS_FC = False


def resolve_default_M(n_features: int, nominal_M: int = DEFAULT_M_NOMINAL) -> int:
    """
    Dataset-independent default policy:
        M = min(n_features, nominal_M)

    The nominal default is 64, while the cap guarantees one or more
    observed features per block and avoids empty latent blocks.
    """
    n_features = int(n_features)
    nominal_M = int(nominal_M)

    if n_features < 1:
        raise ValueError("n_features must be at least 1.")
    if nominal_M < 1:
        raise ValueError("nominal_M must be at least 1.")

    return min(n_features, nominal_M)


# ============================================================
# Small console helper
# ============================================================
def log(msg):
    print(msg, flush=True)


# ============================================================
# Device / seed helpers
# ============================================================
def resolve_torch_device(device_str: str = REQUESTED_TORCH_DEVICE_STR):
    if torch is None:
        return None
    if not torch.cuda.is_available():
        return torch.device("cpu")
    if not device_str.startswith("cuda:"):
        return torch.device(device_str)
    idx = int(device_str.split(":")[1])
    ndev = torch.cuda.device_count()
    if idx >= ndev:
        raise RuntimeError(
            f"Requested {device_str}, but only {ndev} CUDA device(s) are visible."
        )
    return torch.device(device_str)


if torch is not None:
    DEFAULT_TORCH_DEVICE = resolve_torch_device(REQUESTED_TORCH_DEVICE_STR)
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
else:
    DEFAULT_TORCH_DEVICE = None


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cleanup_torch():
    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    gc.collect()


# ============================================================
# 1) Logistic Regression classifier
# ============================================================
def make_clf(seed=0):
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    n_jobs=None,
                    random_state=seed,
                ),
            ),
        ]
    )


# ============================================================
# 2) Classification metric helpers
# ============================================================
def _predict_proba_binary(clf, X):
    if hasattr(clf, "predict_proba"):
        proba = np.asarray(clf.predict_proba(X), dtype=float)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(clf, "decision_function"):
        scores = np.asarray(clf.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-scores))

    raise ValueError("Classifier has neither predict_proba nor decision_function.")


def _safe_binary_auc_or_acc(clf, X_val, y_val):
    y_val = np.asarray(y_val).astype(int)

    if len(np.unique(y_val)) < 2:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))

    try:
        proba = _predict_proba_binary(clf, X_val)
        return float(roc_auc_score(y_val, proba))
    except Exception:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))


# ============================================================
# 3) Strict 5x5 CV with fold-wise generator training
# ============================================================
def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    generator_fit_fn=None,
    n_syn=200,
    do_trtr=True,
    do_c2st=True,
    seed=0,
):
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided.")

    X_real = np.asarray(X_real)
    y_real = np.asarray(y_real).astype(int)

    classes, counts = np.unique(y_real, return_counts=True)
    if len(classes) != 2:
        raise ValueError(
            f"Binary classification requires exactly 2 classes, but found "
            f"{len(classes)}: {classes.tolist()}"
        )
    if counts.min() < 5:
        raise ValueError(
            "RepeatedStratifiedKFold with n_splits=5 requires at least "
            f"5 samples per class. Class counts: "
            f"{dict(zip(classes.tolist(), counts.tolist()))}"
        )

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)
        X_syn_fold = np.nan_to_num(
            np.asarray(X_syn_fold, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_syn_fold = np.asarray(y_syn_fold).astype(int)

        if len(np.unique(y_syn_fold)) < 2:
            raise RuntimeError(
                f"Fold {fold_id}: the synthetic dataset contains fewer than two classes."
            )

        # TSTR: train on synthetic, test on real.
        clf_tstr = make_clf(seed=fold_seed)
        clf_tstr.fit(X_syn_fold, y_syn_fold)
        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)
        tstr_auc.append(float(roc_auc_score(yte, proba_t)))
        tstr_acc.append(float(accuracy_score(yte, pred_t)))

        # TRTR: train on real, test on real.
        if do_trtr:
            clf_trtr = make_clf(seed=fold_seed)
            clf_trtr.fit(Xtr, ytr)
            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)
            trtr_auc.append(float(roc_auc_score(yte, proba_r)))
            trtr_acc.append(float(accuracy_score(yte, pred_r)))

        # C2ST: distinguish real=1 from synthetic=0.
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)]
            )
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf(seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)
            c2st_auc.append(float(roc_auc_score(ymix_te, proba_m)))
            c2st_acc.append(float(accuracy_score(ymix_te, pred_m)))

        log(
            f"[Fold {fold_id:02d}] "
            f"TSTR ACC={tstr_acc[-1]:.4f}, "
            f"AUC={tstr_auc[-1]:.4f}"
        )
        cleanup_torch()

    def _mean_std(values):
        values = np.asarray(values, dtype=float)
        return float(np.mean(values)), float(np.std(values))

    tstr_auc_mean, tstr_auc_std = _mean_std(tstr_auc)
    tstr_acc_mean, tstr_acc_std = _mean_std(tstr_acc)

    out = {
        "tstr_auc_mean": tstr_auc_mean,
        "tstr_auc_std": tstr_auc_std,
        "tstr_acc_mean": tstr_acc_mean,
        "tstr_acc_std": tstr_acc_std,
        "n_folds": len(tstr_auc),
    }

    if do_trtr and len(trtr_auc) > 0:
        trtr_auc_mean, trtr_auc_std = _mean_std(trtr_auc)
        trtr_acc_mean, trtr_acc_std = _mean_std(trtr_acc)
        out.update(
            {
                "trtr_auc_mean": trtr_auc_mean,
                "trtr_auc_std": trtr_auc_std,
                "trtr_acc_mean": trtr_acc_mean,
                "trtr_acc_std": trtr_acc_std,
            }
        )

    if do_c2st and len(c2st_auc) > 0:
        c2st_auc_mean, c2st_auc_std = _mean_std(c2st_auc)
        c2st_acc_mean, c2st_acc_std = _mean_std(c2st_acc)
        out.update(
            {
                "c2st_auc_mean": c2st_auc_mean,
                "c2st_auc_std": c2st_auc_std,
                "c2st_acc_mean": c2st_acc_mean,
                "c2st_acc_std": c2st_acc_std,
            }
        )

    return out


# ============================================================
# 4) CSV saving helpers
# ============================================================
def save_synth_csv(
    X_syn,
    y_syn=None,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_default_cuda0.csv",
    feature_names=None,
    target_col=TARGET_COL,
):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]
    assert len(feature_names) == m

    df = pd.DataFrame(X_syn, columns=feature_names)
    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        df.insert(0, target_col, y_syn)

    df.to_csv(out_csv, index=False)
    log(
        f"Saved CSV: {out_csv} | shape={df.shape} | "
        f"nan_frac={np.isnan(X_syn).mean():.6g}"
    )
    return out_csv


def save_synth_mask_csv(
    R_syn,
    out_csv="MiniBooNE_synth_BSTabDiff_n200_mask_default_cuda0.csv",
):
    R_syn = np.asarray(R_syn).astype(int)
    dfR = pd.DataFrame(R_syn, columns=[f"r{j}" for j in range(R_syn.shape[1])])
    dfR.to_csv(out_csv, index=False)
    log(
        f"Saved mask CSV: {out_csv} | shape={dfR.shape} | "
        f"observed_frac={R_syn.mean():.6g}"
    )
    return out_csv


# ============================================================
# 5) Fixed-default BSTabDiff generator
# ============================================================
def fit_default_bstabdiff(
    Xtr,
    ytr,
    M,
    device=REQUESTED_TORCH_DEVICE_STR,
    seed=0,
    verbose_every=0,
):
    Xtr = np.asarray(Xtr, dtype=np.float32)
    ytr = np.asarray(ytr).astype(int)

    if M < 1 or M > Xtr.shape[1]:
        raise ValueError(
            f"M must satisfy 1 <= M <= number of features ({Xtr.shape[1]}), "
            f"but received M={M}."
        )

    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(Xtr.shape[1])
    ]

    # blocks=None lets BSTabDiff create equal contiguous blocks.
    # No permutation and no GO-BS variant are used.
    gen = fit_block_subunit_generator(
        X=Xtr,
        feature_specs=feature_specs,
        y=ytr,
        M=M,
        blocks=None,
        permute_features=FIXED_PERMUTE_FEATURES,
        prior_type=DEFAULT_PRIOR_TYPE,
        device=device,
        seed=seed,
        prior_epochs=DEFAULT_PRIOR_EPOCHS,
        prior_batch=DEFAULT_PRIOR_BATCH,
        prior_lr=DEFAULT_PRIOR_LR,
        verbose_every=verbose_every,
        save_dir=None,
        save_name=f"bstabdiff_MiniBooNE_default_M{M}_cuda0",
        save_best=True,
        use_ema=DEFAULT_USE_EMA,
        ema_decay=DEFAULT_EMA_DECAY,
        return_train_info=False,
        use_gobs=FIXED_USE_GOBS,
        use_gobs_fc=FIXED_USE_GOBS_FC,
    )
    return gen


def make_default_generator_fit_fn(
    M,
    device=REQUESTED_TORCH_DEVICE_STR,
    verbose_every=0,
):
    def _fit(Xtr, ytr, seed=0):
        return fit_default_bstabdiff(
            Xtr=Xtr,
            ytr=ytr,
            M=M,
            device=device,
            seed=seed,
            verbose_every=verbose_every,
        )

    return _fit


# ============================================================
# 6) Fidelity metric helpers
# ============================================================
def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return 0.0, 0.0

    values = np.concatenate([x, y])
    labels = np.concatenate([np.zeros(nx, dtype=int), np.ones(ny, dtype=int)])
    order = np.argsort(values)
    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1
    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny
    diff = cdf1 - cdf2

    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _rank_transform(X):
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    ranks = np.empty_like(X, dtype=float)
    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=float)
    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centered = X - mean
    var = (centered**2).mean(axis=0)
    eps = 1e-8
    std = np.sqrt(var + eps)
    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)
    return mean, var, skew, kurt


def compute_fidelity_metrics(
    X_real,
    y_real,
    X_syn,
    y_syn,
    feature_names=None,
    generator=None,
    random_state=0,
):
    X_real = np.asarray(X_real, dtype=float)
    X_syn = np.asarray(X_syn, dtype=float)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."
    n_features = X_real.shape[1]

    ks_stats = np.zeros(n_features, dtype=float)
    w1_stats = np.zeros(n_features, dtype=float)
    for j in range(n_features):
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    pearson_real = np.corrcoef(X_real, rowvar=False)
    pearson_syn = np.corrcoef(X_syn, rowvar=False)
    mask_offdiag = ~np.eye(n_features, dtype=bool)
    pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]
    pearson_mean = float(np.mean(pearson_abs_diff))
    pearson_max = float(np.max(pearson_abs_diff))

    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)
    spearman_real = np.corrcoef(ranks_real, rowvar=False)
    spearman_syn = np.corrcoef(ranks_syn, rowvar=False)
    spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]
    spearman_mean = float(np.mean(spearman_abs_diff))
    spearman_max = float(np.max(spearman_abs_diff))

    mi_real = mi_syn = mi_abs = None
    mi_mean = mi_max = None
    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)
        mi_real = mutual_info_classif(X_real, y_real, random_state=random_state)
        mi_syn = mutual_info_classif(X_syn, y_syn, random_state=random_state)
        mi_abs = np.abs(mi_real - mi_syn)
        mi_mean = float(np.mean(mi_abs))
        mi_max = float(np.max(mi_abs))

    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)
    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    ll_real = ll_syn = None
    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    nn_stats = {}
    try:
        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(X_real)
        dists, idx = nn.kneighbors(X_syn, return_distance=True)
        dists = dists.ravel()
        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_real": mi_real,
                "mi_syn": mi_syn,
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )
    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# 7) Load MiniBooNE dataset
# ============================================================
def load_miniboone_dataset(
    csv_path=CSV_PATH,
    label_col=TARGET_COL,
    expected_num_classes=EXPECTED_NUM_CLASSES,
):
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. "
            f"Columns: {list(df.columns)[:10]} ..."
        )

    y_raw = df[label_col].to_numpy()
    X_df = df.drop(columns=[label_col])
    X_df = X_df.select_dtypes(include=[np.number])

    if X_df.shape[1] == 0:
        raise ValueError("No numeric feature columns were found in the dataset.")

    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Encode arbitrary binary labels as 0 and 1.
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, class_names = pd.factorize(y_raw, sort=True)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, class_names = pd.factorize(y_raw, sort=True)
        else:
            y = y_raw.astype(int, copy=False)
            class_names = unique

    y = np.asarray(y, dtype=int)
    classes, counts = np.unique(y, return_counts=True)

    if expected_num_classes is not None and len(classes) != expected_num_classes:
        raise ValueError(
            f"Expected {expected_num_classes} classes, but found {len(classes)}: "
            f"{classes.tolist()}"
        )

    if counts.min() < 5:
        raise ValueError(
            "The requested 5-fold stratified CV requires at least 5 samples "
            f"per class. Class counts: {dict(zip(classes.tolist(), counts.tolist()))}"
        )

    return X, y, list(X_df.columns), np.asarray(class_names)


# ============================================================
# 8) Main
# ============================================================
if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    log(f"Using torch device: {DEFAULT_TORCH_DEVICE}")
    log("Configuration: fixed-default BSTabDiff, no tuning")
    log(f"Nominal default M: {DEFAULT_M_NOMINAL}")
    log(f"Prior type: {DEFAULT_PRIOR_TYPE}")
    log(f"Prior epochs: {DEFAULT_PRIOR_EPOCHS}")
    log(f"Prior batch size: {DEFAULT_PRIOR_BATCH}")
    log(f"Prior learning rate: {DEFAULT_PRIOR_LR}")
    log(f"EMA enabled: {DEFAULT_USE_EMA}")
    log(f"EMA decay: {DEFAULT_EMA_DECAY}")
    log(f"Feature permutation: {FIXED_PERMUTE_FEATURES}")
    log(f"Use GO-BS: {FIXED_USE_GOBS}")
    log(f"Use GO-BS-FC: {FIXED_USE_GOBS_FC}")
    log("Predefined blocks: None (equal contiguous blocks are created internally)")
    log("Classifier: Logistic Regression only")

    # -------- Load MiniBooNE dataset --------
    X, y, feature_names, original_class_names = load_miniboone_dataset(
        csv_path=CSV_PATH,
        label_col=TARGET_COL,
        expected_num_classes=EXPECTED_NUM_CLASSES,
    )

    M = resolve_default_M(X.shape[1])
    class_counts = dict(zip(*np.unique(y, return_counts=True)))

    log(
        f"Loaded {DATASET_NAME}: X.shape={X.shape}, y.shape={y.shape}, "
        f"encoded_classes={np.unique(y).tolist()}, class_counts={class_counts}"
    )
    log(f"Original class values: {original_class_names.tolist()}")
    log(
        f"Resolved default M=min({DEFAULT_M_NOMINAL}, {X.shape[1]})={M}; "
        f"average features per block={X.shape[1] / M:.4f}"
    )

    generator_fit_fn = make_default_generator_fit_fn(
        M=M,
        device=REQUESTED_TORCH_DEVICE_STR,
        verbose_every=0,
    )

    # ============================================================
    # (A) Fit fixed-default BSTabDiff once on full MiniBooNE data
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats(device=DEFAULT_TORCH_DEVICE)
        except Exception:
            pass

    cpu_before = proc.memory_info().rss if proc is not None else 0
    t0 = time.time()
    gen_full = generator_fit_fn(X, y, seed=GLOBAL_SEED)
    gen_train_time = time.time() - t0
    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)

    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated(
            device=DEFAULT_TORCH_DEVICE
        )
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024**3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024**3) if cpu_peak > 0 else 0.0

    log(
        f"\n=== Fixed-default BSTabDiff resource usage "
        f"(trained once on full {DATASET_NAME}, M={M}) ==="
    )
    log(f"  Train time: {gen_train_time:.2f} sec")
    log(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    log(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    # -------- Sample one synthetic dataset --------
    N_SYN = 200
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)

    synth_csv = (
        f"MiniBooNE_synth_BSTabDiff_n{N_SYN}_default_M{M}_cuda0.csv"
    )
    mask_csv = (
        f"MiniBooNE_synth_BSTabDiff_n{N_SYN}_mask_default_M{M}_cuda0.csv"
    )

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv=synth_csv,
        feature_names=feature_names,
        target_col=TARGET_COL,
    )
    save_synth_mask_csv(
        R_syn_global,
        out_csv=mask_csv,
    )

    # ============================================================
    # (B) Strict 5x5 fold-wise evaluation
    # ============================================================
    log(
        "\n=== Evaluating STRICT fixed-default BSTabDiff synthetic + "
        "Logistic Regression (5x5 TSTR/TRTR/C2ST) ==="
    )

    res = eval_5x5_tstr_foldwise_gen(
        X_real=X,
        y_real=y,
        generator_fit_fn=generator_fit_fn,
        n_syn=N_SYN,
        do_trtr=True,
        do_c2st=True,
        seed=0,
    )

    log(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
    log(f"  TSTR AUC: {res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}")
    log(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
    log(f"  TRTR AUC: {res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}")
    log(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
    log(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

    summary_df = pd.DataFrame(
        [
            {
                "Configuration": (
                    f"Default BSTabDiff; M={M}; diffusion; 10000 epochs; "
                    "batch=128; lr=1e-3; EMA=0.999; no permutation; "
                    "no predefined blocks"
                ),
                "Classifier": "Logistic Regression",
                "TSTR ACC (mean ± std)": (
                    f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}"
                ),
                "TSTR AUC (mean ± std)": (
                    f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}"
                ),
                "TRTR ACC (mean ± std)": (
                    f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}"
                ),
                "TRTR AUC (mean ± std)": (
                    f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}"
                ),
                "C2ST ACC (mean ± std)": (
                    f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}"
                ),
                "C2ST AUC (mean ± std)": (
                    f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}"
                ),
            }
        ]
    )

    summary_csv = (
        f"bstabdiff_MiniBooNE_cv_summary_lr_default_M{M}_cuda0.csv"
    )
    summary_df.to_csv(summary_csv, index=False)

    log(
        f"\n=== {DATASET_NAME}: fixed-default BSTabDiff + "
        "Logistic Regression summary ==="
    )
    log(summary_df.to_markdown(index=False))
    log(f"Saved CV summary: {summary_csv}")

    # ============================================================
    # (C) Fidelity diagnostics
    # ============================================================
    fidelity = compute_fidelity_metrics(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        feature_names=feature_names,
        generator=gen_full,
        random_state=0,
    )

    log(f"\n=== {DATASET_NAME}: Fidelity metrics (real vs synthetic) ===")
    ks_mean = float(np.mean(fidelity["ks_per_feature"]))
    ks_max = float(np.max(fidelity["ks_per_feature"]))
    w1_mean = float(np.mean(fidelity["wasserstein_per_feature"]))
    w1_max = float(np.max(fidelity["wasserstein_per_feature"]))

    log(f"  KS statistic per feature:    mean={ks_mean:.4f}, max={ks_max:.4f}")
    log(f"  Wasserstein-1 per feature:   mean={w1_mean:.4f}, max={w1_max:.4f}")
    log(
        f"  Pearson |corr_real - corr_syn|: "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )
    log(
        f"  Spearman |corr_real - corr_syn|: "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]
    log("  Higher-order moments (mean abs diff over features):")
    log(f"    mean:  {higher['mean_abs_diff']:.4e}")
    log(f"    var:   {higher['var_abs_diff']:.4e}")
    log(f"    skew:  {higher['skew_abs_diff']:.4e}")
    log(f"    kurt:  {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        log(
            f"  Mutual information (feature-label) |ΔMI|: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    if "loglik_real_mean" in fidelity:
        log(
            f"  Log-likelihood (if available): "
            f"real={fidelity['loglik_real_mean']}, "
            f"synthetic={fidelity['loglik_syn_mean']}"
        )

    log(
        f"  NaN fraction (real): {fidelity['nan_frac_real']:.4e}, "
        f"NaN fraction (synthetic): {fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]
        log("  Privacy (NN distance from synthetic to nearest real):")
        log(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

Using torch device: cuda:0
Configuration: fixed-default BSTabDiff, no tuning
Nominal default M: 64
Prior type: diffusion
Prior epochs: 10000
Prior batch size: 128
Prior learning rate: 0.001
EMA enabled: True
EMA decay: 0.999
Feature permutation: False
Use GO-BS: False
Use GO-BS-FC: False
Predefined blocks: None (equal contiguous blocks are created internally)
Classifier: Logistic Regression only
Loaded MiniBooNE: X.shape=(130064, 50), y.shape=(130064,), encoded_classes=[0, 1], class_counts={np.int64(0): np.int64(93565), np.int64(1): np.int64(36499)}
Original class values: [0, 1]
Resolved default M=min(64, 50)=50; average features per block=1.0000

=== Fixed-default BSTabDiff resource usage (trained once on full MiniBooNE, M=50) ===
  Train time: 35.57 sec
  Approx. peak GPU memory: 0.096 GiB
  Approx. CPU RSS at end: 1.603 GiB
Saved CSV: MiniBooNE_synth_BSTabDiff_n200_default_M50_cuda0.csv | shape=(200, 51) | nan_frac=0
Saved mask CSV: MiniBooNE_synth_BSTabDiff_n200_mask_default_M50_cu